# Notebook 18: Crop Recommendation Intelligence

## 1. Objective

The objective of this notebook is to develop a data-driven crop recommendation system that ranks candidate crops for each district–season combination using historical agricultural data and the selected yield prediction model developed in the previous modeling stage.

The system will estimate the expected yield of eligible crop candidates under specified district, season, and year conditions, then rank those candidates based on their predicted yield. Historical performance and supporting evidence will be retained to help interpret the recommendations.

## 2. Scope

This notebook will:

1. Load the existing engineered agricultural dataset and saved yield prediction artifacts.
2. Validate the required features, target definition, data schema, and artifact compatibility.
3. Identify eligible crop candidates using available historical evidence.
4. Construct district–season–crop combinations for a clearly specified prediction year.
5. Generate yield predictions using the existing selected model without retraining it by default.
6. Establish a historical-performance baseline for comparison.
7. Rank candidate crops by predicted yield and retain relevant historical evidence.
8. Evaluate ranking behavior, investigate weakly supported recommendations, and document limitations.
9. Export reproducible recommendation outputs and metadata for downstream analysis.

## 3. Recommendation Definition

A recommendation is defined as the relative ranking of eligible crop candidates for a particular district and season, for a specified prediction year.

The primary ranking signal will be predicted crop yield. Historical observations and evidence coverage will be used to contextualize the rankings and identify candidates for which the available evidence is limited.

A higher predicted yield does not necessarily imply higher profitability, lower agricultural risk, better water efficiency, or superior agronomic suitability. These claims will not be made without the required data and validation.

## 4. Data and Model Dependencies

This notebook is expected to reuse:

- The cleaned and engineered agricultural dataset produced by the preceding data preparation notebooks.
- The selected yield prediction model and its metadata from the existing modeling workflow.
- Relevant historical crop, season, district, year, and yield observations.

Actual artifact paths, available columns, category coverage, and model input requirements will be verified before implementation. No dataset paths, features, or model capabilities will be assumed without checking the existing artifacts.

## 5. Methodology

The workflow will proceed through data and artifact validation, candidate eligibility definition, district–season candidate generation, yield prediction, historical baseline construction, candidate ranking, ranking evaluation, error and limitation analysis, and output export.

Evaluation will respect the temporal structure of the data. Previously inspected holdout observations will not be represented as an untouched final test set.

## 6. Expected Outputs

The notebook is expected to produce:

- `crop_recommendations_ranked.csv` — ranked crop candidates by district and season.
- `crop_recommendation_evaluation.json` — evaluation results and validation details.
- `crop_recommendation_metadata.json` — model reference, prediction year, features, assumptions, and limitations.

Existing project output conventions and file paths will be checked before these artifacts are saved to avoid accidental overwrites or duplicate outputs.

## 7. Constraints and Limitations

- The selected yield model will be reused; retraining is outside the default scope of this notebook.
- Existing cleaning, exploratory data analysis, preprocessing, and feature engineering will not be repeated.
- Candidates with insufficient historical evidence or unsupported feature categories will be identified rather than presented as equally reliable recommendations.
- Predictions will be interpreted as model estimates, not causal evidence or guarantees of future yield.
- Production API implementation and application integration will be handled separately.
- Agricultural risk intelligence, detailed explainability, scenario analysis, and model registry outputs remain assigned to their respective downstream notebooks.

## 8. Completion Criteria

This notebook will be considered complete when candidate generation and ranking are reproducible, predictions satisfy the model input contract, historical evidence is available alongside rankings, evaluation limitations are documented, and the required outputs are saved for downstream use.

**Final objective:** Produce a transparent, reproducible, and evidence-aware crop ranking system for district–season combinations without duplicating earlier work or overstating what the available data can establish.

In [2]:
# Locate the project root reliably
_current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [_current_path, *_current_path.parents]
        if (path / "notebooks").is_dir()
        and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing both "
        "'notebooks' and 'models' directories."
    )

YIELD_MODEL_DIR = PROJECT_ROOT / "models" / "yield_prediction"

YIELD_MODEL_PATH = (
    YIELD_MODEL_DIR / "individual_crop_hgb_location.joblib"
)

YIELD_MODEL_METADATA_PATH = (
    YIELD_MODEL_DIR / "individual_crop_hgb_location_metadata.json"
)

CROP_RECOMMENDATION_DIR = (
    PROJECT_ROOT / "models" / "crop_recommendation"
)

print(f"Corrected project root: {PROJECT_ROOT}")
print(f"Yield model exists: {YIELD_MODEL_PATH.is_file()}")
print(f"Model metadata exists: {YIELD_MODEL_METADATA_PATH.is_file()}")

Corrected project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Yield model exists: True
Model metadata exists: True


In [3]:
# ============================================================
# CELL 02: INSPECT EXISTING PROJECT ARTIFACTS
# ============================================================

print("Relevant project files:\n")

SEARCH_DIRS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "notebooks",
]

for search_dir in SEARCH_DIRS:
    print(f"\n--- {search_dir.relative_to(PROJECT_ROOT)} ---")

    if not search_dir.is_dir():
        print("Directory not found.")
        continue

    for path in sorted(search_dir.rglob("*")):
        if not path.is_file():
            continue

        # Skip generated environments and cache directories.
        if any(
            part in {"__pycache__", ".ipynb_checkpoints", ".git"}
            for part in path.parts
        ):
            continue

        if path.suffix.lower() in {
            ".csv", ".parquet", ".xlsx", ".xls",
            ".joblib", ".pkl", ".json", ".ipynb"
        }:
            try:
                size_mb = path.stat().st_size / (1024 ** 2)
                print(f"{path.relative_to(PROJECT_ROOT)} ({size_mb:.2f} MB)")
            except OSError:
                print(path.relative_to(PROJECT_ROOT))

Relevant project files:


--- data ---
data\processed\historical_food_grains_apy_2013-14_to_2024-25.csv (5.70 MB)
data\processed\historical_food_grains_apy_cleaned_flagged.csv (15.20 MB)
data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv (41.53 MB)
data\processed\weather_pilot_2013_2024\nasa_power_weather_pilot_2013_2024.csv (3.56 MB)
data\processed\weather_regional\apy_districts_without_weather_coverage.csv (0.01 MB)
data\processed\weather_regional\apy_weather_integrated_exploratory.csv (115.81 MB)
data\processed\weather_regional\apy_weather_integration_summary.csv (0.00 MB)
data\processed\weather_regional\apy_weather_proxy_district_year_coverage.csv (0.00 MB)
data\processed\weather_regional\crop_weather_integration_validation_summary.csv (0.00 MB)
data\processed\weather_regional\district_weather_annual_2013_2024.csv (2.01 MB)
data\processed\weather_regional\district_weather_annual_cleaned_flagged_2013_2024.csv (2.23 MB)
data\processed\weather_regional\district_weat

In [4]:
# ============================================================
# CELL 03: INSPECT CANDIDATE DATASETS
# ============================================================

CANDIDATE_DATASETS = {
    "apy_long": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
    ),
    "apy_cleaned": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "historical_food_grains_apy_cleaned_flagged.csv"
    ),
    "apy_weather_integrated": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "weather_regional"
        / "apy_weather_integrated_exploratory.csv"
    ),
}

dataset_inspection = []

for name, path in CANDIDATE_DATASETS.items():
    print(f"\n{'=' * 70}")
    print(f"Dataset: {name}")
    print(f"Path: {path.relative_to(PROJECT_ROOT)}")

    if not path.is_file():
        print("STATUS: File not found")
        continue

    sample = pd.read_csv(path, nrows=5)

    # Read the header separately; avoid loading large datasets into memory.
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    dataset_inspection.append({
        "dataset": name,
        "exists": True,
        "sample_rows": len(sample),
        "column_count": len(columns),
        "has_target": TARGET_COLUMN in columns,
        "missing_model_features": [
            col for col in REQUIRED_FEATURES if col not in columns
        ],
        "columns": columns,
    })

    print(f"Columns: {len(columns)}")
    print(f"Contains target '{TARGET_COLUMN}': {TARGET_COLUMN in columns}")
    print(
        "Missing model features:",
        [col for col in REQUIRED_FEATURES if col not in columns],
    )
    display(sample.head(3))

print("\nDataset inspection summary:")
display(pd.DataFrame([
    {
        "dataset": item["dataset"],
        "exists": item["exists"],
        "column_count": item["column_count"],
        "has_target": item["has_target"],
        "missing_model_features": item["missing_model_features"],
    }
    for item in dataset_inspection
]))


Dataset: apy_long
Path: data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv
Columns: 13
Contains target 'Yield': True
Missing model features: ['FE_YearStart']


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True



Dataset: apy_cleaned
Path: data\processed\historical_food_grains_apy_cleaned_flagged.csv
Columns: 89
Contains target 'Yield': False
Missing model features: ['FE_YearStart']


,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,Area-2015-16,Production-2015-16,Yield-2015-16,Area-2016-17,Production-2016-17,Yield-2016-17,Area-2017-18,Production-2017-18,Yield-2017-18,Area-2018-19,Production-2018-19,Yield-2018-19,Area-2019-20,Production-2019-20,Yield-2019-20,Area-2020-21,Production-2020-21,Yield-2020-21,Area-2021-22,Production-2021-22,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25,Crop_Category,QC-2013-14-AreaZero-ProductionPositive,QC-2013-14-AreaPositive-ProductionZero,QC-2013-14-NegativeMeasurements,QC-2013-14-PositiveYieldWithoutPositiveAreaProduction,QC-2014-15-AreaZero-ProductionPositive,QC-2014-15-AreaPositive-ProductionZero,QC-2014-15-NegativeMeasurements,QC-2014-15-PositiveYieldWithoutPositiveAreaProduction,QC-2015-16-AreaZero-ProductionPositive,QC-2015-16-AreaPositive-ProductionZero,QC-2015-16-NegativeMeasurements,QC-2015-16-PositiveYieldWithoutPositiveAreaProduction,QC-2016-17-AreaZero-ProductionPositive,QC-2016-17-AreaPositive-ProductionZero,QC-2016-17-NegativeMeasurements,QC-2016-17-PositiveYieldWithoutPositiveAreaProduction,QC-2017-18-AreaZero-ProductionPositive,QC-2017-18-AreaPositive-ProductionZero,QC-2017-18-NegativeMeasurements,QC-2017-18-PositiveYieldWithoutPositiveAreaProduction,QC-2018-19-AreaZero-ProductionPositive,QC-2018-19-AreaPositive-ProductionZero,QC-2018-19-NegativeMeasurements,QC-2018-19-PositiveYieldWithoutPositiveAreaProduction,QC-2019-20-AreaZero-ProductionPositive,QC-2019-20-AreaPositive-ProductionZero,QC-2019-20-NegativeMeasurements,QC-2019-20-PositiveYieldWithoutPositiveAreaProduction,QC-2020-21-AreaZero-ProductionPositive,QC-2020-21-AreaPositive-ProductionZero,QC-2020-21-NegativeMeasurements,QC-2020-21-PositiveYieldWithoutPositiveAreaProduction,QC-2021-22-AreaZero-ProductionPositive,QC-2021-22-AreaPositive-ProductionZero,QC-2021-22-NegativeMeasurements,QC-2021-22-PositiveYieldWithoutPositiveAreaProduction,QC-2022-23-AreaZero-ProductionPositive,QC-2022-23-AreaPositive-ProductionZero,QC-2022-23-NegativeMeasurements,QC-2022-23-PositiveYieldWithoutPositiveAreaProduction,QC-2023-24-AreaZero-ProductionPositive,QC-2023-24-AreaPositive-ProductionZero,QC-2023-24-NegativeMeasurements,QC-2023-24-PositiveYieldWithoutPositiveAreaProduction,QC-2024-25-AreaZero-ProductionPositive,QC-2024-25-AreaPositive-ProductionZero,QC-2024-25-NegativeMeasurements,QC-2024-25-PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,0.0,0.0,2240.0,0.0,0.0,1430.0,0.0,0.0,530.0,0.0,0.0,1466.0,0.0,0.0,1387.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0,Grouping — Verify Source Definition,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,0.0,0.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Grouping — Verify Source Definition,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,0.0,0.0,2127.0,0.0,0.0,1430.0,0.0,0.0,530.0,0.0,0.0,1466.0,0.0,0.0,1387.0,NaN,NaN,NaN,0.0,0.0,1133.0,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0,Grouping — Verify Source Definition,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,Fals


Dataset: apy_weather_integrated
Path: data\processed\weather_regional\apy_weather_integrated_exploratory.csv
Columns: 39
Contains target 'Yield': True
Missing model features: ['FE_YearStart']


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction,apy_geography_id,start_year,weather_proxy_calendar_year,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance,weather_proxy_available,_merge,apy_measurements_complete,weather_and_apy_complete,weather_alignment_method,weather_full_season_alignment_verified,weather_available,weather_match_status
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True,APY-GEO-0001,2013,2013.0,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.0,365.0,365.0,365.0,False,False,False,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True,APY-GEO-0001,2013,2013.0,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.0,365.0,365.0,365.0,False,False,False,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True,APY-GEO-0001,2013,2013.0,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.0,365.0,365.0,365.0,False,False,False,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED



Dataset inspection summary:


,dataset,exists,column_count,has_target,missing_model_features
0,apy_long,True,13,True,[FE_YearStart]
1,apy_cleaned,True,89,False,[FE_YearStart]
2,apy_weather_integrated,True,39,True,[FE_YearStart]


In [5]:
# ============================================================
# CELL 04: VALIDATE YEAR MAPPING AND DATA QUALITY
# ============================================================

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_weather_integrated_exploratory.csv"
)

# Read only the columns needed for this inspection.
inspection_columns = [
    "State",
    "District",
    "Crop",
    "Season",
    "Crop_Category",
    "Year",
    "Yield",
    "start_year",
    "weather_proxy_calendar_year",
    "weather_available",
    "weather_match_status",
]

df_inspection = pd.read_csv(
    DATA_PATH,
    usecols=lambda column: column in inspection_columns,
)

print("Dataset shape:", df_inspection.shape)

print("\nYear-field comparison:")
display(
    df_inspection[
        ["Year", "start_year", "weather_proxy_calendar_year"]
    ].head(10)
)

print("\nYear-field missing values:")
display(
    df_inspection[
        ["Year", "start_year", "weather_proxy_calendar_year"]
    ].isna().sum().to_frame("missing_count")
)

print("\nYear-field data types:")
display(
    df_inspection[
        ["Year", "start_year", "weather_proxy_calendar_year"]
    ].dtypes.to_frame("dtype")
)

print("\nStart-year summary:")
display(df_inspection["start_year"].describe())

print("\nYield summary:")
display(df_inspection["Yield"].describe())

print("\nDistinct crop categories:")
display(df_inspection["Crop_Category"].value_counts(dropna=False).head(20))

print("\nSeason values:")
display(df_inspection["Season"].value_counts(dropna=False))

print("\nWeather match status:")
display(df_inspection["weather_match_status"].value_counts(dropna=False))

Dataset shape: (376272, 11)

Year-field comparison:


,Year,start_year,weather_proxy_calendar_year
0,2013-14,2013,2013.0
1,2013-14,2013,2013.0
2,2013-14,2013,2013.0
3,2013-14,2013,2013.0
4,2013-14,2013,2013.0
5,2013-14,2013,2013.0
6,2013-14,2013,2013.0
7,2013-14,2013,2013.0
8,2013-14,2013,2013.0
9,2013-14,2013,2013.0



Year-field missing values:


,missing_count
Year,0
start_year,0
weather_proxy_calendar_year,56148



Year-field data types:


,dtype
Year,object
start_year,int64
weather_proxy_calendar_year,float64


... (6 outputs left out)


Season values:


Season
Total     143028
Kharif    106692
Rabi       88704
Summer     37848
Name: count, dtype: int64


Weather match status:


weather_match_status
MATCHED                320124
NO_WEATHER_COVERAGE     56148
Name: count, dtype: int64

In [6]:
# ============================================================
# CELL 05: VERIFY YEAR MAPPING AND CANDIDATE ELIGIBILITY
# ============================================================

# 1. Verify that start_year matches the beginning of Year.
year_start_from_label = (
    df_inspection["Year"]
    .astype(str)
    .str.extract(r"^(\d{4})", expand=False)
    .astype("Int64")
)

year_mapping_check = pd.DataFrame({
    "start_year": df_inspection["start_year"],
    "year_start_from_label": year_start_from_label,
})

print("Year mapping mismatch count:")
print(
    (
        year_mapping_check["start_year"]
        != year_mapping_check["year_start_from_label"]
    ).sum()
)

print("\nYear mapping examples:")
display(
    year_mapping_check.drop_duplicates().sort_values("start_year").head(15)
)

# 2. Inspect yield quality by the dataset's crop category.
category_yield_quality = (
    df_inspection
    .assign(
        yield_missing=df_inspection["Yield"].isna(),
        yield_zero=df_inspection["Yield"].eq(0),
        yield_negative=df_inspection["Yield"].lt(0),
        yield_positive=df_inspection["Yield"].gt(0),
    )
    .groupby("Crop_Category", dropna=False)
    .agg(
        rows=("Yield", "size"),
        missing_yield=("yield_missing", "sum"),
        zero_yield=("yield_zero", "sum"),
        negative_yield=("yield_negative", "sum"),
        positive_yield=("yield_positive", "sum"),
        distinct_crops=("Crop", "nunique"),
        distinct_districts=("District", "nunique"),
    )
    .reset_index()
)

print("\nYield quality by crop category:")
display(category_yield_quality)

# 3. Inspect eligible individual-crop records and candidate keys.
individual_crop_rows = df_inspection.loc[
    df_inspection["Crop_Category"].eq("Individual Crop Candidate")
].copy()

candidate_key = ["State", "District", "Crop", "Season", "Year"]

print("\nIndividual-crop rows:", len(individual_crop_rows))
print(
    "Rows with missing key fields:",
    individual_crop_rows[candidate_key].isna().any(axis=1).sum(),
)
print(
    "Duplicate candidate keys:",
    individual_crop_rows.duplicated(candidate_key, keep=False).sum(),
)

print("\nMost frequent duplicate candidate keys, if any:")
duplicate_rows = individual_crop_rows.loc[
    individual_crop_rows.duplicated(candidate_key, keep=False)
].sort_values(candidate_key)

display(duplicate_rows.head(20))

Year mapping mismatch count:
0

Year mapping examples:


,start_year,year_start_from_label
0,2013,2013
31356,2014,2014
62712,2015,2015
94068,2016,2016
125424,2017,2017
156780,2018,2018
188136,2019,2019
219492,2020,2020
250848,2021,2021
282204,2022,2022



Yield quality by crop category:


,Crop_Category,rows,missing_yield,zero_yield,negative_yield,positive_yield,distinct_crops,distinct_districts
0,Aggregate Total — Exclude from Individual-Crop...,42072,4962,14,0,37096,2,748
1,Grouping — Verify Source Definition,107640,23112,41,0,84487,4,747
2,Individual Crop Candidate,226560,66959,362,0,159239,13,748



Individual-crop rows: 226560
Rows with missing key fields: 0
Duplicate candidate keys: 0

Most frequent duplicate candidate keys, if any:


,State,District,Crop,Season,Crop_Category,Year,Yield,start_year,weather_proxy_calendar_year,weather_available,weather_match_status


In [7]:
# ============================================================
# CELL 06: DEFINE ELIGIBLE HISTORICAL CROP RECORDS
# ============================================================

import numpy as np

# Read the full integrated dataset only after confirming its schema.
df_apy = pd.read_csv(DATA_PATH)

required_columns = [
    "State",
    "District",
    "Crop",
    "Season",
    "Crop_Category",
    "Year",
    "Yield",
    "start_year",
]

missing_columns = [
    column for column in required_columns
    if column not in df_apy.columns
]

if missing_columns:
    raise ValueError(
        f"Integrated dataset is missing required columns: {missing_columns}"
    )

# Confirm the year feature mapping before applying it.
year_start = (
    df_apy["Year"]
    .astype(str)
    .str.extract(r"^(\d{4})", expand=False)
    .astype("Int64")
)

if not df_apy["start_year"].astype("Int64").equals(year_start):
    raise ValueError("Year-to-start_year mapping is inconsistent.")

df_apy["FE_YearStart"] = df_apy["start_year"].astype("int64")

# Keep only rows explicitly categorized as individual crop candidates.
individual_mask = df_apy["Crop_Category"].eq(
    "Individual Crop Candidate"
)

individual_crop_data = df_apy.loc[individual_mask].copy()

# Separate missing, zero, non-finite, and positive yields.
yield_values = pd.to_numeric(
    individual_crop_data["Yield"],
    errors="coerce",
)

finite_yield = np.isfinite(yield_values)

quality_summary = pd.Series({
    "individual_crop_rows": len(individual_crop_data),
    "missing_yield": yield_values.isna().sum(),
    "non_finite_yield": (~finite_yield & yield_values.notna()).sum(),
    "zero_yield": yield_values.eq(0).sum(),
    "negative_yield": yield_values.lt(0).sum(),
    "positive_finite_yield": (
        finite_yield & yield_values.gt(0)
    ).sum(),
})

print("Individual-crop yield quality:")
display(quality_summary.to_frame("count"))

# Positive finite yields form the initial historical-evidence dataset.
valid_yield_mask = finite_yield & yield_values.gt(0)

df_historical_valid = individual_crop_data.loc[
    valid_yield_mask
].copy()

df_historical_valid["Yield"] = yield_values.loc[
    valid_yield_mask
].astype(float)

print("\nEligible historical dataset shape:", df_historical_valid.shape)
print("Year range:", (
    df_historical_valid["FE_YearStart"].min(),
    df_historical_valid["FE_YearStart"].max(),
))
print("Distinct states:", df_historical_valid["State"].nunique())
print("Distinct districts:", df_historical_valid["District"].nunique())
print("Distinct crops:", df_historical_valid["Crop"].nunique())
print("Distinct seasons:", df_historical_valid["Season"].nunique())

print("\nYield quantiles for inspection:")
display(
    df_historical_valid["Yield"]
    .quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1])
    .to_frame("Yield")
)

print("\nLargest observed yields:")
display(
    df_historical_valid[
        ["State", "District", "Crop", "Season", "Year", "Yield"]
    ]
    .nlargest(10, "Yield")
)

Individual-crop yield quality:


,count
individual_crop_rows,226560
missing_yield,66959
non_finite_yield,0
zero_yield,362
negative_yield,0
positive_finite_yield,159239



Eligible historical dataset shape: (159239, 40)
Year range: (np.int64(2013), np.int64(2024))
Distinct states: 34
Distinct districts: 748
Distinct crops: 13
Distinct seasons: 4

Yield quantiles for inspection:


,Yield
0.00,1.0
0.01,184.0
0.25,720.0
0.50,1172.0
0.75,2231.0
0.99,6497.0
1.00,385540.0



Largest observed yields:


,State,District,Crop,Season,Year,Yield
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,385540.0
313625,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,385540.0
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,105588.0
313610,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,105588.0
288727,Goa,North Goa,Rice,Kharif,2022-23,44815.0
288745,Goa,South Goa,Rice,Kharif,2022-23,39221.0
213450,Telangana,Mulugu,Tur,Kharif,2019-20,22846.0
213452,Telangana,Mulugu,Tur,Total,2019-20,22846.0
182809,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,Jowar,Kharif,2018-19,22000.0
182810,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,Jowar,Total,2018-19,22000.0


In [8]:
# ============================================================
# CELL 07: INVESTIGATE EXTREME YIELDS AND SEASON SEMANTICS
# ============================================================

# 1. Inspect the upper tail of the yield distribution.
yield_thresholds = df_historical_valid["Yield"].quantile(
    [0.95, 0.99, 0.995, 0.999]
)

print("Yield thresholds:")
display(yield_thresholds.to_frame("Yield"))

extreme_threshold = yield_thresholds.loc[0.99]

extreme_yields = df_historical_valid.loc[
    df_historical_valid["Yield"] > extreme_threshold
].copy()

print(f"\nRecords above the 99th percentile ({extreme_threshold:.2f}):")
print("Count:", len(extreme_yields))

print("\nExtreme yields by season:")
display(
    extreme_yields.groupby("Season")
    .agg(
        records=("Yield", "size"),
        median_yield=("Yield", "median"),
        max_yield=("Yield", "max"),
    )
    .sort_values("records", ascending=False)
)

print("\nExtreme yields by crop:")
display(
    extreme_yields.groupby("Crop")
    .agg(
        records=("Yield", "size"),
        median_yield=("Yield", "median"),
        max_yield=("Yield", "max"),
    )
    .sort_values("max_yield", ascending=False)
)

# 2. Compare season distributions across the eligible data.
print("\nAll eligible records by season:")
display(
    df_historical_valid.groupby("Season")
    .agg(
        records=("Yield", "size"),
        median_yield=("Yield", "median"),
        p99_yield=("Yield", lambda x: x.quantile(0.99)),
        max_yield=("Yield", "max"),
    )
    .sort_values("records", ascending=False)
)

# 3. Inspect the largest records with source-quality context.
context_columns = [
    "State",
    "District",
    "Crop",
    "Season",
    "Year",
    "Area",
    "Production",
    "Yield",
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_NegativeMeasurements",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
]

available_context_columns = [
    col for col in context_columns
    if col in df_historical_valid.columns
]

print("\nLargest yields with quality indicators:")
display(
    df_historical_valid[available_context_columns]
    .nlargest(20, "Yield")
)

# 4. Check how often each district-crop-year has both a named season
#    and a Total record. This is descriptive, not an assumption that
#    Total is necessarily invalid.
base_keys = ["State", "District", "Crop", "Year"]

season_counts = (
    df_historical_valid.groupby(base_keys)["Season"]
    .agg(
        season_count="nunique",
        has_total=lambda x: x.eq("Total").any(),
        has_named_season=lambda x: x.isin(
            ["Kharif", "Rabi", "Summer"]
        ).any(),
    )
    .reset_index()
)

print("\nDistrict-crop-year season structure:")
display(
    season_counts[
        ["season_count", "has_total", "has_named_season"]
    ].value_counts(dropna=False).to_frame("groups")
)

Yield thresholds:


,Yield
0.950,4022.000
0.990,6497.000
0.995,7572.050
0.999,9553.382



Records above the 99th percentile (6497.00):
Count: 1591

Extreme yields by season:


,records,median_yield,max_yield
Season,,,
Rabi,622,7495.0,385540.0
Total,538,7577.0,385540.0
Kharif,336,7789.5,44815.0
Summer,95,7410.0,21300.0



Extreme yields by crop:


,records,median_yield,max_yield
Crop,,,
Urad,4,197395.0,385540.0
Moong,4,57157.0,105588.0
Rice,12,12880.0,44815.0
Tur,6,9059.0,22846.0
Jowar,32,7238.0,22000.0
Ragi,5,7553.0,21300.0
Maize,1524,7565.0,17941.0
Lentil,2,16880.0,16880.0
Small Millets,2,15261.0,15261.0


... (2 outputs left out)


Largest yields with quality indicators:


,State,District,Crop,Season,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.00,0.00,385540.0,False,False,False,True
313625,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,0.00,0.00,385540.0,False,False,False,True
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.00,0.00,105588.0,False,False,False,True
313610,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,0.00,0.00,105588.0,False,False,False,True
288727,Goa,North Goa,Rice,Kharif,2022-23,0.01,0.47,44815.0,False,False,False,False
288745,Goa,South Goa,Rice,Kharif,2022-23,0.01,0.47,39221.0,False,False,False,False
213450,Telangana,Mulugu,Tur,Kharif,2019-20,0.00,0.01,22846.0,True,False,False,True
213452,Telangana,Mulugu,Tur,Total,2019-20,0.00,0.01,22846.0,True,False,False,True
182809,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,Jowar,Kharif,2018-19,0.00,0.00,22000.0,False,False,False,True
182810,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,Jowar,Total,2018-19,0.00,0.00,22000.0,False,False,False,True



District-crop-year season structure:


,,,groups
season_count,has_total,has_named_season,
2,True,True,49960
3,True,True,14225
4,True,True,4161


In [9]:
# ============================================================
# CELL 08: QUANTIFY SOURCE-QUALITY FLAGS
# ============================================================

quality_flags = [
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_NegativeMeasurements",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
]

available_flags = [
    col for col in quality_flags
    if col in df_historical_valid.columns
]

print("Available quality flags:", available_flags)

# Ensure flags are interpreted as Boolean values.
flag_data = df_historical_valid[available_flags].copy()

for col in available_flags:
    if flag_data[col].dtype != bool:
        flag_data[col] = (
            flag_data[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(False)
            .astype(bool)
        )

# Individual flag counts and overlaps.
flag_summary = pd.DataFrame({
    "flagged_rows": flag_data.sum(),
    "flagged_percent": flag_data.mean().mul(100),
})

print("\nFlag counts and percentages:")
display(flag_summary)

print("\nOverlapping flag combinations:")
display(
    flag_data.value_counts(dropna=False)
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

# Compare yield distributions before and after excluding the
# positive-yield-without-valid-area/production flag.
quality_flag = "QC_PositiveYieldWithoutPositiveAreaProduction"

if quality_flag in available_flags:
    flagged_mask = flag_data[quality_flag]

    quality_comparison = pd.DataFrame([
        {
            "subset": "All positive finite yields",
            "rows": len(df_historical_valid),
            "median_yield": df_historical_valid["Yield"].median(),
            "p99_yield": df_historical_valid["Yield"].quantile(0.99),
            "max_yield": df_historical_valid["Yield"].max(),
        },
        {
            "subset": "Flagged by QC_PositiveYieldWithoutPositiveAreaProduction",
            "rows": int(flagged_mask.sum()),
            "median_yield": df_historical_valid.loc[
                flagged_mask, "Yield"
            ].median(),
            "p99_yield": df_historical_valid.loc[
                flagged_mask, "Yield"
            ].quantile(0.99),
            "max_yield": df_historical_valid.loc[
                flagged_mask, "Yield"
            ].max(),
        },
        {
            "subset": "Not flagged by that quality rule",
            "rows": int((~flagged_mask).sum()),
            "median_yield": df_historical_valid.loc[
                ~flagged_mask, "Yield"
            ].median(),
            "p99_yield": df_historical_valid.loc[
                ~flagged_mask, "Yield"
            ].quantile(0.99),
            "max_yield": df_historical_valid.loc[
                ~flagged_mask, "Yield"
            ].max(),
        },
    ])

    print("\nYield distribution comparison:")
    display(quality_comparison)

# Inspect Total-season records and how frequently they coexist
# with named-season records for the same district/crop/year.
print("\nSeason distribution in positive-yield data:")
display(
    df_historical_valid["Season"]
    .value_counts(dropna=False)
    .to_frame("rows")
)

total_records = df_historical_valid.loc[
    df_historical_valid["Season"].eq("Total")
]

print("\nTotal-season records:", len(total_records))
print(
    "Total-season records with the positive-yield quality flag:",
    (
        total_records[quality_flag].astype(bool).sum()
        if quality_flag in total_records.columns
        else "Flag unavailable"
    ),
)

Available quality flags: ['QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_NegativeMeasurements', 'QC_PositiveYieldWithoutPositiveAreaProduction']

Flag counts and percentages:


,flagged_rows,flagged_percent
QC_AreaZero-ProductionPositive,5225,3.281231
QC_AreaPositive-ProductionZero,4787,3.006173
QC_NegativeMeasurements,0,0.000000
QC_PositiveYieldWithoutPositiveAreaProduction,68159,42.802957



Overlapping flag combinations:


,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction,rows
0,False,False,False,False,91080
1,False,False,False,True,58147
2,True,False,False,True,5225
3,False,True,False,True,4787



Yield distribution comparison:


,subset,rows,median_yield,p99_yield,max_yield
0,All positive finite yields,159239,1172.0,6497.0,385540.0
1,Flagged by QC_PositiveYieldWithoutPositiveArea...,68159,935.0,5110.0,385540.0
2,Not flagged by that quality rule,91080,1531.0,6997.0,44815.0



Season distribution in positive-yield data:


,rows
Season,
Total,68346
Kharif,45783
Rabi,34178
Summer,10932



Total-season records: 68346
Total-season records with the positive-yield quality flag: 26911


In [10]:
# ============================================================
# CELL 09: CREATE PROVISIONAL QUALITY-CONTROLLED DATA
# ============================================================

# These source flags are used to identify observations requiring
# caution. Preserve all source rows in df_apy for traceability.
quality_flag_columns = [
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_NegativeMeasurements",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
]

missing_flag_columns = [
    col for col in quality_flag_columns
    if col not in df_apy.columns
]

if missing_flag_columns:
    raise ValueError(
        f"Expected QC flags are missing: {missing_flag_columns}"
    )

# Normalize flags to Boolean values without changing the source file.
for col in quality_flag_columns:
    if df_apy[col].dtype != bool:
        normalized = (
            df_apy[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False})
        )

        if normalized.isna().any():
            raise ValueError(
                f"Unexpected values found in quality flag: {col}"
            )

        df_apy[col] = normalized.astype(bool)

# Keep individual-crop candidates only.
df_individual_all = df_apy.loc[
    df_apy["Crop_Category"].eq("Individual Crop Candidate")
].copy()

# Require positive finite yield for historical performance evidence.
yield_numeric = pd.to_numeric(
    df_individual_all["Yield"],
    errors="coerce",
)

positive_finite_yield = (
    np.isfinite(yield_numeric) & yield_numeric.gt(0)
)

# Preserve an audit table for records not eligible for primary evidence.
any_quality_flag = df_individual_all[
    quality_flag_columns
].any(axis=1)

df_historical_flagged = df_individual_all.loc[
    positive_finite_yield & any_quality_flag
].copy()

# Primary evidence: positive finite yield and no available QC flag.
primary_mask = positive_finite_yield & ~any_quality_flag

df_historical_primary = df_individual_all.loc[
    primary_mask
].copy()

df_historical_primary["Yield"] = yield_numeric.loc[
    primary_mask
].astype(float)

# Total is not included in the individual-season candidate pool.
df_historical_primary_named_seasons = (
    df_historical_primary.loc[
        df_historical_primary["Season"].isin(
            ["Kharif", "Rabi", "Summer"]
        )
    ].copy()
)

print("Provisional quality-control summary:")
display(pd.DataFrame([
    {
        "dataset": "All individual-crop rows",
        "rows": len(df_individual_all),
    },
    {
        "dataset": "Positive finite yield, any QC flag",
        "rows": len(df_historical_flagged),
    },
    {
        "dataset": "Positive finite yield, no QC flags",
        "rows": len(df_historical_primary),
    },
    {
        "dataset": "Primary evidence, named seasons only",
        "rows": len(df_historical_primary_named_seasons),
    },
]))

print("\nPrimary evidence yield distribution:")
display(
    df_historical_primary_named_seasons["Yield"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999])
    .to_frame("Yield")
)

print("\nRows by season in primary evidence:")
display(
    df_historical_primary_named_seasons["Season"]
    .value_counts(dropna=False)
    .to_frame("rows")
)

print("\nCheck duplicate historical keys:")
history_key = ["State", "District", "Crop", "Season", "Year"]
duplicate_count = df_historical_primary_named_seasons.duplicated(
    history_key,
    keep=False,
).sum()

print("Duplicate rows across historical keys:", duplicate_count)

Provisional quality-control summary:


,dataset,rows
0,All individual-crop rows,226560
1,"Positive finite yield, any QC flag",68159
2,"Positive finite yield, no QC flags",91080
3,"Primary evidence, named seasons only",49645



Primary evidence yield distribution:


,Yield
count,49645.000000
mean,1982.061577
std,1467.266403
min,26.000000
50%,1606.000000
90%,3784.000000
95%,4500.000000
99%,7286.000000
99.9%,10074.356000
max,44815.000000



Rows by season in primary evidence:


,rows
Season,
Kharif,25489
Rabi,19195
Summer,4961



Check duplicate historical keys:
Duplicate rows across historical keys: 0


In [11]:
# ============================================================
# CELL 10: INSPECT SAVED YIELD MODEL METADATA
# ============================================================

if not YIELD_MODEL_METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Model metadata not found: {YIELD_MODEL_METADATA_PATH}"
    )

with open(YIELD_MODEL_METADATA_PATH, "r", encoding="utf-8") as file:
    yield_model_metadata = json.load(file)

print("Metadata path:", YIELD_MODEL_METADATA_PATH)
print("\nMetadata keys:")
display(pd.DataFrame({
    "key": list(yield_model_metadata.keys()),
    "value_type": [
        type(value).__name__
        for value in yield_model_metadata.values()
    ],
}))

print("\nFull saved model metadata:")
print(json.dumps(yield_model_metadata, indent=2, default=str))

Metadata path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location_metadata.json

Metadata keys:


,key,value_type
0,model_name,str
1,project,str
2,candidate_scope,str
3,features,list
4,categorical_features,list
5,numeric_features,list
6,target,str
7,training_rows,int
8,training_year_start,int
9,training_year_end,int



Full saved model metadata:
{
  "model_name": "HistGradientBoostingRegressor",
  "project": "BharatCrop Intelligence",
  "candidate_scope": "Individual Crop Candidate cohort",
  "features": [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District",
    "FE_YearStart"
  ],
  "categorical_features": [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District"
  ],
  "numeric_features": [
    "FE_YearStart"
  ],
  "target": "Yield",
  "training_rows": 35784,
  "training_year_start": 2013,
  "training_year_end": 2022,
  "holdout_rows": 7369,
  "holdout_evaluated_during_final_fit": false,
  "validation_windows": [
    {
      "train_through": 2018,
      "validation_years": [
        2019,
        2020
      ]
    },
    {
      "train_through": 2020,
      "validation_years": [
        2021,
        2022
      ]
    }
  ],
  "validation_metrics": {
    "2019-2020": {
      "MAE": 448.306,
      "RMSE": 672.961,
      "R2": 0.798
    },
    "2021-2022": {


In [12]:
# ============================================================
# CELL 11: VERIFY MODEL CONTRACT AND PREDICTION-YEAR INPUTS
# ============================================================

import joblib

PREDICTION_YEAR = 2025

# 1. Validate the saved model feature contract.
metadata_features = yield_model_metadata.get("features", [])
metadata_categorical = yield_model_metadata.get("categorical_features", [])
metadata_numeric = yield_model_metadata.get("numeric_features", [])
metadata_target = yield_model_metadata.get("target")

if metadata_features != REQUIRED_FEATURES:
    raise ValueError(
        "Notebook feature contract differs from saved model metadata.\n"
        f"Notebook: {REQUIRED_FEATURES}\n"
        f"Metadata: {metadata_features}"
    )

if metadata_target != TARGET_COLUMN:
    raise ValueError(
        f"Target mismatch: notebook={TARGET_COLUMN}, "
        f"metadata={metadata_target}"
    )

if set(metadata_categorical + metadata_numeric) != set(metadata_features):
    raise ValueError(
        "Categorical/numeric feature lists do not cover the model features."
    )

print("Model feature contract verified.")
print("Prediction year:", PREDICTION_YEAR)
print("Training period:", (
    yield_model_metadata["training_year_start"],
    yield_model_metadata["training_year_end"],
))
print(
    "Prediction year is beyond training period:",
    PREDICTION_YEAR > yield_model_metadata["training_year_end"],
)

# 2. Inspect available categories and location coverage.
# The 2025 rows are prediction candidates, not training observations.
print("\nHistorical evidence year range:")
print((
    df_historical_primary_named_seasons["FE_YearStart"].min(),
    df_historical_primary_named_seasons["FE_YearStart"].max(),
))

print("\nCandidate crop counts in primary historical evidence:")
display(
    df_historical_primary_named_seasons["Crop"]
    .value_counts()
    .rename_axis("Crop")
    .to_frame("historical_rows")
)

print("\nSeason counts in primary historical evidence:")
display(
    df_historical_primary_named_seasons["Season"]
    .value_counts()
    .rename_axis("Season")
    .to_frame("historical_rows")
)

# 3. Identify which 2024 district/crop/season combinations have
#    prior evidence and which locations/crops have training-period support.
history_keys = ["State", "District", "Crop", "Season"]

evidence_counts = (
    df_historical_primary_named_seasons
    .groupby(history_keys)
    .agg(
        historical_rows=("Yield", "size"),
        first_year=("FE_YearStart", "min"),
        last_year=("FE_YearStart", "max"),
        median_historical_yield=("Yield", "median"),
    )
    .reset_index()
)

print("\nHistorical district-crop-season evidence summary:")
display(evidence_counts["historical_rows"].describe())

print("\nEvidence rows with no observations in the model training period:")
print(
    (
        evidence_counts["last_year"]
        < yield_model_metadata["training_year_start"]
    ).sum()
)

print("\nModel artifact exists:", YIELD_MODEL_PATH.is_file())
print("Model metadata features:", metadata_features)
print("Categorical features:", metadata_categorical)
print("Numeric features:", metadata_numeric)

Model feature contract verified.
Prediction year: 2025
Training period: (2013, 2022)
Prediction year is beyond training period: True

Historical evidence year range:
(np.int64(2013), np.int64(2024))

Candidate crop counts in primary historical evidence:


,historical_rows
Crop,
Rice,11143
Maize,8355
Wheat,4969
Urad,4162
Gram,3426
Moong,3321
Tur,3149
Jowar,2907
Bajra,2336



Season counts in primary historical evidence:


,historical_rows
Season,
Kharif,25489
Rabi,19195
Summer,4961



Historical district-crop-season evidence summary:


count    6220.000000
mean        7.981511
std         4.273472
min         1.000000
25%         3.000000
50%        10.000000
75%        12.000000
max        12.000000
Name: historical_rows, dtype: float64


Evidence rows with no observations in the model training period:
0

Model artifact exists: True
Model metadata features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Categorical features: ['Crop', 'Season', 'Crop_Category', 'State', 'District']
Numeric features: ['FE_YearStart']


In [13]:
# ============================================================
# CELL 12: LOAD MODEL AND VERIFY PREDICTION BEHAVIOR
# ============================================================

if not YIELD_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Saved model not found: {YIELD_MODEL_PATH}"
    )

yield_model = joblib.load(YIELD_MODEL_PATH)

# Build a small input batch using the exact saved model features.
diagnostic_features = df_historical_primary_named_seasons[
    REQUIRED_FEATURES
].head(20).copy()

if diagnostic_features.isna().any().any():
    raise ValueError(
        "Diagnostic model input contains missing feature values."
    )

print("Loaded model type:", type(yield_model).__name__)
print("Diagnostic input shape:", diagnostic_features.shape)
print("Diagnostic input columns:", diagnostic_features.columns.tolist())

# Generate predictions using the saved artifact.
diagnostic_predictions = np.asarray(
    yield_model.predict(diagnostic_features),
    dtype=float,
)

if len(diagnostic_predictions) != len(diagnostic_features):
    raise ValueError("Prediction count does not match input row count.")

if not np.isfinite(diagnostic_predictions).all():
    raise ValueError("Model returned non-finite diagnostic predictions.")

diagnostic_results = diagnostic_features.copy()
diagnostic_results["observed_yield"] = (
    df_historical_primary_named_seasons["Yield"]
    .head(20)
    .to_numpy()
)
diagnostic_results["predicted_yield"] = diagnostic_predictions
diagnostic_results["prediction_error"] = (
    diagnostic_results["predicted_yield"]
    - diagnostic_results["observed_yield"]
)

print("\nDiagnostic prediction results:")
display(diagnostic_results)

print("\nPrediction summary:")
display(
    pd.Series(diagnostic_predictions, name="predicted_yield")
    .describe()
)

Loaded model type: Pipeline
Diagnostic input shape: (20, 6)
Diagnostic input columns: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']

Diagnostic prediction results:


,Crop,Season,Crop_Category,State,District,FE_YearStart,observed_yield,predicted_yield,prediction_error
32,Rice,Kharif,Individual Crop Candidate,Andaman And Nicobar Islands,North And Middle Andaman,2013,2010.0,2001.558015,-8.441985
147,Bajra,Kharif,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,921.0,1160.175169,239.175169
156,Gram,Rabi,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,800.0,751.743720,-48.256280
158,Jowar,Kharif,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,673.0,1015.779341,342.779341
159,Jowar,Rabi,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,475.0,1107.238838,632.238838
161,Maize,Kharif,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,1965.0,3133.626375,1168.626375
162,Maize,Rabi,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,7454.0,5915.504627,-1538.495373
174,Ragi,Kharif,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,1539.0,1016.172798,-522.827202
175,Ragi,Rabi,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,1779.0,1244.992481,-534.007519
177,Rice,Kharif,Individual Crop Candidate,Andhra Pradesh,Ananthapuramu,2013,2112.0,2488.724616,376.724616



Prediction summary:


count      20.000000
mean     2117.176288
std      1692.001831
min       503.623146
25%      1016.074434
50%      1478.924068
75%      2801.601323
max      6549.141368
Name: predicted_yield, dtype: float64

In [14]:
# ============================================================
# CELL 13: INSPECT EXISTING MODEL-SELECTION EVIDENCE
# ============================================================

print("Yield model directory:", YIELD_MODEL_DIR)
print("\nExisting files:")

for path in sorted(YIELD_MODEL_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(PROJECT_ROOT)}")

print("\nJSON report/metadata summaries:")

for path in sorted(YIELD_MODEL_DIR.rglob("*.json")):
    print(f"\n--- {path.relative_to(PROJECT_ROOT)} ---")

    try:
        with path.open("r", encoding="utf-8") as file:
            payload = json.load(file)

        if isinstance(payload, dict):
            for key, value in payload.items():
                if isinstance(value, (dict, list)):
                    print(f"{key}: {json.dumps(value, ensure_ascii=False)[:1800]}")
                else:
                    print(f"{key}: {value}")
        else:
            print(json.dumps(payload, ensure_ascii=False)[:3000])

    except (OSError, json.JSONDecodeError) as exc:
        print("Could not read JSON:", exc)

Yield model directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction

Existing files:
  models\yield_prediction\individual_crop_hgb_location.joblib
  models\yield_prediction\individual_crop_hgb_location_metadata.json
  models\yield_prediction\yield_model_deployment_readiness_report.json

JSON report/metadata summaries:

--- models\yield_prediction\individual_crop_hgb_location_metadata.json ---
model_name: HistGradientBoostingRegressor
project: BharatCrop Intelligence
candidate_scope: Individual Crop Candidate cohort
features: ["Crop", "Season", "Crop_Category", "State", "District", "FE_YearStart"]
categorical_features: ["Crop", "Season", "Crop_Category", "State", "District"]
numeric_features: ["FE_YearStart"]
target: Yield
training_rows: 35784
training_year_start: 2013
training_year_end: 2022
holdout_rows: 7369
holdout_evaluated_during_final_fit: False
validation_windows: [{"train_through": 2018, "validation_years": [2019, 2020]}, {"train_through": 202

In [15]:
# ============================================================
# CELL 14: LOCATE EXISTING NOTEBOOK 16 EVALUATION EVIDENCE
# ============================================================

notebook_dir = PROJECT_ROOT / "notebooks"

candidate_notebooks = [
    notebook_dir / "16_yield_prediction_baselines.ipynb",
    notebook_dir / "17_yield_model_training.ipynb",
]

for notebook_path in candidate_notebooks:
    print(f"\n{'=' * 75}")
    print("Notebook:", notebook_path)
    print("Exists:", notebook_path.is_file())

    if not notebook_path.is_file():
        continue

    with notebook_path.open("r", encoding="utf-8") as file:
        notebook_content = json.load(file)

    for cell_index, cell in enumerate(notebook_content.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        outputs = cell.get("outputs", [])

        output_text = []
        for output in outputs:
            text_value = output.get("text", [])
            if isinstance(text_value, list):
                output_text.extend(text_value)

            for item in output.get("data", {}).values():
                if isinstance(item, list):
                    output_text.extend(
                        value for value in item
                        if isinstance(value, str)
                    )

        combined_output = "\n".join(output_text)

        keywords = (
            "model_comparison",
            "comparison_results",
            "candidate_metrics",
            "validation_metrics",
            "MAE",
            "RMSE",
            "HistGradientBoostingRegressor",
            "best_model",
            "selected_model",
        )

        if any(word.lower() in
               (source + "\n" + combined_output).lower()
               for word in keywords):
            print(f"\n--- Cell {cell_index} ---")
            print("CODE:")
            print(source[:2500])
            if combined_output.strip():
                print("\nOUTPUT:")
                print(combined_output[:2500])


Notebook: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\notebooks\16_yield_prediction_baselines.ipynb
Exists: False

Notebook: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\notebooks\17_yield_model_training.ipynb
Exists: False


In [16]:
# ============================================================
# CELL 15: LOCATE EXISTING MODELING NOTEBOOKS
# ============================================================

for search_root in [
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT,
]:
    if not search_root.is_dir():
        print(f"Directory not found: {search_root}")
        continue

    print(f"\nSearching: {search_root}")

    matches = sorted(
        path for path in search_root.rglob("*.ipynb")
        if ".ipynb_checkpoints" not in str(path)
        and path != Path.cwd()
    )

    for path in matches:
        print(" ", path.relative_to(PROJECT_ROOT))

    if not matches:
        print("  No notebooks found.")


Searching: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\notebooks
  notebooks\01_project_setup_and_source_registry.ipynb
  notebooks\02_apy_dataset_acquisition_and_audit.ipynb
  notebooks\03_weather_dataset_acquisition_and_audit.ipynb
  notebooks\04_apy_data_overview.ipynb
  notebooks\05_apy_data_cleaning.ipynb
  notebooks\06_apy_data_preprocessing.ipynb
  notebooks\07_apy_eda_analysis.ipynb
  notebooks\08_apy_data_visualization.ipynb
  notebooks\09_weather_data_overview.ipynb
  notebooks\10_weather_data_cleaning.ipynb
  notebooks\11_weather_data_preprocessing.ipynb
  notebooks\12_weather_eda_analysis.ipynb
  notebooks\13_weather_data_visualization.ipynb
  notebooks\14_crop_weather_integration.ipynb
  notebooks\15_feature_engineering.ipynb
  notebooks\16_yield_model_development_and_selection.ipynb
  notebooks\17_yield_model_deployment_readiness.ipynb
  notebooks\18_crop_recommendation.ipynb

Searching: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
  notebooks\01_p

In [17]:
# ============================================================
# CELL 16: INSPECT NOTEBOOK 16 MODEL-SELECTION EVIDENCE
# ============================================================

source_notebook = (
    PROJECT_ROOT
    / "notebooks"
    / "16_yield_model_development_and_selection.ipynb"
)

if not source_notebook.is_file():
    raise FileNotFoundError(
        f"Model-development notebook not found: {source_notebook}"
    )

with source_notebook.open("r", encoding="utf-8") as file:
    source_content = json.load(file)

keywords = (
    "model_comparison",
    "comparison_results",
    "candidate_metrics",
    "validation_metrics",
    "mean_absolute_error",
    "mean_squared_error",
    "r2_score",
    "HistGradientBoostingRegressor",
    "RandomForestRegressor",
    "ExtraTreesRegressor",
    "best_model",
    "selected_model",
    "model selection",
    "cross-validation",
    "validation",
)

matches_found = 0

for cell_index, cell in enumerate(source_content.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matched = any(
        keyword.lower() in source.lower()
        for keyword in keywords
    )

    output_parts = []
    for output in cell.get("outputs", []):
        text_output = output.get("text", [])
        if isinstance(text_output, list):
            output_parts.extend(text_output)

        for mime, value in output.get("data", {}).items():
            if mime in ("text/plain", "text/html"):
                if isinstance(value, list):
                    output_parts.extend(value)
                elif isinstance(value, str):
                    output_parts.append(value)

    output_text = "\n".join(output_parts)

    if not matched and not output_text.strip():
        continue

    matches_found += 1
    print(f"\n{'=' * 75}")
    print(f"CELL {cell_index}")

    if matched:
        print("\nCODE:")
        print(source[:1800])

    if output_text.strip():
        print("\nSAVED OUTPUT:")
        print(output_text[:2500])

if matches_found == 0:
    print(
        "No relevant code or saved outputs were found. "
        "We will inspect the notebook's Markdown and cell structure next."
    )


CELL 1

SAVED OUTPUT:
Project root exists: True

Weather-regional directory exists: True



Required files:



Integrated dataset

  Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_weather_integrated_exploratory.csv

  Exists: True



Historical APY long dataset

  Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv

  Exists: True



Relevant saved files in weather_regional:

  apy_districts_without_weather_coverage.csv

  apy_weather_integrated_exploratory.csv

  apy_weather_integration_summary.csv

  apy_weather_proxy_district_year_coverage.csv

  crop_weather_integration_validation_summary.csv

  district_weather_annual_2013_2024.csv

  district_weather_annual_cleaned_flagged_2013_2024.csv

  district_weather_annual_preprocessed_2013_2024.csv


CELL 2

SAVED OUTPUT:
Loaded integrated dataset: (376272, 39)



Reconstructed cohorts:

Primary:      107,267

In [18]:
# CELL 17 — INSPECT FINAL MODEL-SELECTION DECISION

source_notebook = (
    PROJECT_ROOT
    / "notebooks"
    / "16_yield_model_development_and_selection.ipynb"
)

with source_notebook.open("r", encoding="utf-8") as file:
    notebook_data = json.load(file)

for i, cell in enumerate(notebook_data["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    source_lower = source.lower()

    if any(term in source_lower for term in [
        "selected_model",
        "selection rationale",
        "final model",
        "deployment",
        "save",
        "joblib.dump",
        "model comparison",
        "final decision",
        "conclusion",
    ]):
        print(f"\n{'=' * 65}")
        print(f"CELL {i}")
        print(source[:1400])

        for output in cell.get("outputs", []):
            text_output = output.get("text", [])
            if text_output:
                print("\nOUTPUT:")
                print("".join(text_output)[:1200])

            data = output.get("data", {})
            plain = data.get("text/plain", [])
            if plain:
                print("\nSAVED RESULT:")
                print(
                    "".join(plain)
                    if isinstance(plain, list)
                    else plain
                )


CELL 1
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "weather_regional"

required_files = {
    "Integrated dataset": (
        DATA_DIR / "apy_weather_integrated_exploratory.csv"
    ),
    "Historical APY long dataset": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
    ),
}

print("Project root exists:", PROJECT_ROOT.exists())
print("Weather-regional directory exists:", DATA_DIR.exists())

print("\nRequired files:")
for label, path in required_files.items():
    print(f"\n{label}")
    print("  Path:", path)
    print("  Exists:", path.exists())

if DATA_DIR.exists():
    print("\nRelevant saved files in weather_regional:")
    for path in sorted(DATA_DIR.glob("*")):
        if path.is_file() and (
            "integrated" in path.name.lower()
            or "consistency" in path.name.lo

In [19]:

# CELL 18 — CHECK HISTORICAL RANKING-EVALUATION COVERAGE
# Read-only diagnostic. Does not train or modify any model.

evaluation_years = [2023, 2024]

coverage_df = df_historical_primary_named_seasons.copy()

coverage_df["FE_YearStart"] = pd.to_numeric(
    coverage_df["start_year"], errors="coerce"
)

coverage_df = coverage_df[
    coverage_df["FE_YearStart"].isin(evaluation_years)
].copy()

coverage_df["Yield"] = pd.to_numeric(
    coverage_df["Yield"], errors="coerce"
)

coverage_df = coverage_df[
    np.isfinite(coverage_df["Yield"])
    & (coverage_df["Yield"] > 0)
].copy()

group_columns = ["State", "District", "Season", "FE_YearStart"]

coverage_summary = (
    coverage_df.groupby(group_columns)
    .agg(
        candidate_crops=("Crop", "nunique"),
        observed_rows=("Crop", "size"),
    )
    .reset_index()
)

print("Evaluation years:", evaluation_years)
print("Eligible historical rows:", len(coverage_df))
print("District-season-year groups:", len(coverage_summary))
print(
    "Groups with at least 2 crops:",
    int((coverage_summary["candidate_crops"] >= 2).sum()),
)
print(
    "Groups with at least 3 crops:",
    int((coverage_summary["candidate_crops"] >= 3).sum()),
)

print("\nCandidate crops per group:")
display(coverage_summary["candidate_crops"].describe().to_frame())

print("\nGroups by evaluation year:")
display(
    coverage_summary.groupby("FE_YearStart").agg(
        groups=("candidate_crops", "size"),
        groups_with_2plus_crops=(
            "candidate_crops",
            lambda s: int((s >= 2).sum()),
        ),
        groups_with_3plus_crops=(
            "candidate_crops",
            lambda s: int((s >= 3).sum()),
        ),
    )
)


Evaluation years: [2023, 2024]
Eligible historical rows: 8433
District-season-year groups: 3319
Groups with at least 2 crops: 2365
Groups with at least 3 crops: 1478

Candidate crops per group:


,candidate_crops
count,3319.000000
mean,2.540826
std,1.434970
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,9.000000



Groups by evaluation year:


,groups,groups_with_2plus_crops,groups_with_3plus_crops
FE_YearStart,,,
2023,1661,1175,725
2024,1658,1190,753


In [33]:

# CELL 19 — PREPARE PREDICTIONS FOR HISTORICAL RANKING EVALUATION
# Read-only with respect to the saved model and source data.

import joblib
import json
import numpy as np
import pandas as pd

# Load the existing selected model and its metadata.
yield_model = joblib.load(YIELD_MODEL_PATH)

with YIELD_MODEL_METADATA_PATH.open("r", encoding="utf-8") as file:
    yield_model_metadata = json.load(file)

model_features = yield_model_metadata["features"]

if model_features != [
    "Crop", "Season", "Crop_Category", "State", "District", "FE_YearStart"
]:
    raise ValueError(
        f"Unexpected saved-model feature contract: {model_features}"
    )

ranking_df = coverage_df.copy()

# Verify that every model feature is available.
missing_features = [
    feature for feature in model_features
    if feature not in ranking_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing model features in evaluation data: {missing_features}"
    )

# Predict on the same historical rows whose observed yields we'll compare.
X_eval = ranking_df.loc[:, model_features].copy()

predictions = np.asarray(yield_model.predict(X_eval), dtype=float)

if len(predictions) != len(ranking_df):
    raise ValueError("Prediction count does not match evaluation rows.")

if not np.isfinite(predictions).all():
    raise ValueError("Model produced non-finite predictions.")

ranking_df["Predicted_Yield"] = predictions

# A crop should occur at most once within a district-season-year group.
duplicate_keys = ranking_df.duplicated(
    ["State", "District", "Season", "FE_YearStart", "Crop"],
    keep=False,
)

if duplicate_keys.any():
    display(
        ranking_df.loc[
            duplicate_keys,
            ["State", "District", "Season", "FE_YearStart", "Crop", "Yield"]
        ].head(20)
    )
    raise ValueError(
        "Duplicate crop rows found within evaluation groups; resolve this "
        "before calculating ranking metrics."
    )

print("Saved model loaded:", type(yield_model).__name__)
print("Model training years:", yield_model_metadata["training_year_start"],
      "to", yield_model_metadata["training_year_end"])
print("Evaluation years:", sorted(ranking_df["FE_YearStart"].unique()))
print("Rows with predictions:", len(ranking_df))
print("Predictions finite:", np.isfinite(predictions).all())
print("Prediction range:", (
    round(float(predictions.min()), 3),
    round(float(predictions.max()), 3),
))

display(
    ranking_df[
        ["State", "District", "Season", "FE_YearStart",
         "Crop", "Yield", "Predicted_Yield"]
    ].head(10)
)


Saved model loaded: Pipeline
Model training years: 2013 to 2022
Evaluation years: [np.int64(2023), np.int64(2024)]
Rows with predictions: 8433
Predictions finite: True
Prediction range: (136.012, 8557.189)


,State,District,Season,FE_YearStart,Crop,Yield,Predicted_Yield
313592,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2023,Rice,2100.0,2435.288112
313634,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2023,Maize,3438.0,4294.018903
313646,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2023,Ragi,1269.0,1552.285840
313648,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2023,Rice,1762.0,3449.962015
313649,Andhra Pradesh,Alluri Sitharama Raju,Rabi,2023,Rice,3180.0,3790.162458
313654,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2023,Small Millets,542.0,1263.598534
313664,Andhra Pradesh,Alluri Sitharama Raju,Rabi,2023,Urad,885.0,979.127852
313666,Andhra Pradesh,Anakapalli,Kharif,2023,Bajra,855.0,1915.124765
313679,Andhra Pradesh,Anakapalli,Rabi,2023,Moong,634.0,1008.993716
313690,Andhra Pradesh,Anakapalli,Kharif,2023,Rice,2589.0,3449.962015


In [21]:

# CELL 20 — EVALUATE HISTORICAL CROP RANKING QUALITY
# Retrospective diagnostic, not an independent untouched test.

from scipy.stats import spearmanr

group_columns = ["State", "District", "Season", "FE_YearStart"]

ranking_results = []

for group_key, group in ranking_df.groupby(group_columns, sort=False):
    if len(group) < 2:
        continue

    # Observed winner(s): allow ties.
    best_actual = group["Yield"].max()
    actual_winners = set(
        group.loc[group["Yield"].eq(best_actual), "Crop"]
    )

    # Predicted ordering, descending.
    predicted_order = group.sort_values(
        "Predicted_Yield",
        ascending=False,
        kind="mergesort",
    )

    predicted_top1 = predicted_order.iloc[0]["Crop"]
    predicted_top3 = set(predicted_order.head(3)["Crop"])

    # Spearman correlation requires variation in both rankings.
    correlation = spearmanr(
        group["Predicted_Yield"],
        group["Yield"],
    ).statistic

    ranking_results.append({
        "State": group_key[0],
        "District": group_key[1],
        "Season": group_key[2],
        "FE_YearStart": group_key[3],
        "Candidate_Count": len(group),
        "Top1_Hit": predicted_top1 in actual_winners,
        "Top3_Hit": bool(predicted_top3 & actual_winners),
        "Spearman": correlation,
        "Actual_Winner_Count": len(actual_winners),
    })

ranking_metrics_df = pd.DataFrame(ranking_results)

if ranking_metrics_df.empty:
    raise ValueError("No groups with at least two crops were evaluable.")

print("Evaluated groups:", len(ranking_metrics_df))
print(
    "Top-1 hit rate:",
    round(ranking_metrics_df["Top1_Hit"].mean(), 4),
)
print(
    "Top-3 hit rate:",
    round(ranking_metrics_df["Top3_Hit"].mean(), 4),
)
print(
    "Mean Spearman correlation:",
    round(ranking_metrics_df["Spearman"].mean(), 4),
)
print(
    "Median Spearman correlation:",
    round(ranking_metrics_df["Spearman"].median(), 4),
)
print(
    "Groups with undefined Spearman:",
    int(ranking_metrics_df["Spearman"].isna().sum()),
)

print("\nMetrics by evaluation year:")
display(
    ranking_metrics_df.groupby("FE_YearStart").agg(
        groups=("Candidate_Count", "size"),
        mean_candidates=("Candidate_Count", "mean"),
        top1_hit_rate=("Top1_Hit", "mean"),
        top3_hit_rate=("Top3_Hit", "mean"),
        mean_spearman=("Spearman", "mean"),
        median_spearman=("Spearman", "median"),
    ).round(4)
)

print("\nMetrics by candidate-set size:")
ranking_metrics_df["Candidate_Bucket"] = pd.cut(
    ranking_metrics_df["Candidate_Count"],
    bins=[1, 2, 3, 5, 9],
    labels=["2 crops", "3 crops", "4–5 crops", "6–9 crops"],
    include_lowest=True,
)

display(
    ranking_metrics_df.groupby(
        "Candidate_Bucket",
        observed=False,
    ).agg(
        groups=("Candidate_Count", "size"),
        top1_hit_rate=("Top1_Hit", "mean"),
        top3_hit_rate=("Top3_Hit", "mean"),
        mean_spearman=("Spearman", "mean"),
    ).round(4)
)


C:\Users\gsr58\AppData\Local\Temp\ipykernel_18396\3896367647.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation = spearmanr(


Evaluated groups: 2365
Top-1 hit rate: 0.792
Top-3 hit rate: 0.9983
Mean Spearman correlation: 0.792
Median Spearman correlation: 1.0
Groups with undefined Spearman: 1

Metrics by evaluation year:


,groups,mean_candidates,top1_hit_rate,top3_hit_rate,mean_spearman,median_spearman
FE_YearStart,,,,,,
2023,1175,3.1583,0.7889,0.9983,0.7821,1.0
2024,1190,3.1664,0.7950,0.9983,0.8018,1.0



Metrics by candidate-set size:


,groups,top1_hit_rate,top3_hit_rate,mean_spearman
Candidate_Bucket,,,,
2 crops,887,0.8658,1.0000,0.7314
3 crops,716,0.7849,1.0000,0.8219
4–5 crops,629,0.7266,0.9968,0.8360
6–9 crops,133,0.6466,0.9850,0.8266


In [22]:

# CELL 21 — COMPARE MODEL AGAINST HISTORICAL CROP-SEASON BASELINE
# Baseline uses only eligible observations from years before 2023.

baseline_train = df_historical_primary_named_seasons.copy()
baseline_train["FE_YearStart"] = pd.to_numeric(
    baseline_train["start_year"], errors="coerce"
)
baseline_train["Yield"] = pd.to_numeric(
    baseline_train["Yield"], errors="coerce"
)

baseline_train = baseline_train[
    (baseline_train["FE_YearStart"] <= 2022)
    & np.isfinite(baseline_train["Yield"])
    & (baseline_train["Yield"] > 0)
].copy()

crop_season_medians = (
    baseline_train.groupby(["Crop", "Season"])["Yield"]
    .median()
    .rename("Baseline_Predicted_Yield")
    .reset_index()
)

baseline_eval = ranking_df.merge(
    crop_season_medians,
    on=["Crop", "Season"],
    how="left",
    validate="many_to_one",
)

missing_baseline = baseline_eval["Baseline_Predicted_Yield"].isna()

print("Baseline training rows:", len(baseline_train))
print("Evaluation rows:", len(baseline_eval))
print("Rows without baseline predictions:", int(missing_baseline.sum()))

if missing_baseline.any():
    display(
        baseline_eval.loc[
            missing_baseline, ["Crop", "Season"]
        ].drop_duplicates().sort_values(["Season", "Crop"])
    )
    raise ValueError(
        "Some evaluation crop-season pairs have no pre-2023 baseline."
    )

def evaluate_group_rankings(
    data,
    prediction_column,
    group_columns=("State", "District", "Season", "FE_YearStart"),
):
    results = []

    for group_key, group in data.groupby(list(group_columns), sort=False):
        if len(group) < 2:
            continue

        actual_max = group["Yield"].max()
        actual_winners = set(
            group.loc[group["Yield"].eq(actual_max), "Crop"]
        )

        # Stable tie-breaking for predicted yields.
        ordered = group.sort_values(
            prediction_column,
            ascending=False,
            kind="mergesort",
        )

        top1 = ordered.iloc[0]["Crop"]
        top3 = set(ordered.head(3)["Crop"])

        rho = spearmanr(
            group[prediction_column],
            group["Yield"],
        ).statistic

        results.append({
            "FE_YearStart": group_key[3],
            "Candidate_Count": len(group),
            "Top1_Hit": top1 in actual_winners,
            "Top3_Hit": bool(top3 & actual_winners),
            "Spearman": rho,
        })

    return pd.DataFrame(results)

model_metrics = evaluate_group_rankings(
    ranking_df, "Predicted_Yield"
)
baseline_metrics = evaluate_group_rankings(
    baseline_eval, "Baseline_Predicted_Yield"
)

def summarize_metrics(metrics, method_name):
    return {
        "Method": method_name,
        "Groups": len(metrics),
        "Top1_Hit_Rate": metrics["Top1_Hit"].mean(),
        "Top3_Hit_Rate": metrics["Top3_Hit"].mean(),
        "Mean_Spearman": metrics["Spearman"].mean(),
        "Defined_Spearman_Groups": int(metrics["Spearman"].notna().sum()),
    }

comparison_df = pd.DataFrame([
    summarize_metrics(model_metrics, "Saved HGB model"),
    summarize_metrics(baseline_metrics, "Crop-Season median baseline"),
])

display(comparison_df.round(4))

print("\nTop-1 hit rate by year:")
yearly_comparison = []

for year in sorted(ranking_df["FE_YearStart"].unique()):
    yearly_comparison.append({
        "Year": int(year),
        "Model_Top1": model_metrics.loc[
            model_metrics["FE_YearStart"].eq(year), "Top1_Hit"
        ].mean(),
        "Baseline_Top1": baseline_metrics.loc[
            baseline_metrics["FE_YearStart"].eq(year), "Top1_Hit"
        ].mean(),
        "Model_Spearman": model_metrics.loc[
            model_metrics["FE_YearStart"].eq(year), "Spearman"
        ].mean(),
        "Baseline_Spearman": baseline_metrics.loc[
            baseline_metrics["FE_YearStart"].eq(year), "Spearman"
        ].mean(),
    })

display(pd.DataFrame(yearly_comparison).round(4))


Baseline training rows: 41212
Evaluation rows: 8433
Rows without baseline predictions: 0


C:\Users\gsr58\AppData\Local\Temp\ipykernel_18396\3759436355.py:74: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(
C:\Users\gsr58\AppData\Local\Temp\ipykernel_18396\3759436355.py:74: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(


,Method,Groups,Top1_Hit_Rate,Top3_Hit_Rate,Mean_Spearman,Defined_Spearman_Groups
0,Saved HGB model,2365,0.792,0.9983,0.7920,2364
1,Crop-Season median baseline,2365,0.693,0.9987,0.7009,2364



Top-1 hit rate by year:


,Year,Model_Top1,Baseline_Top1,Model_Spearman,Baseline_Spearman
0,2023,0.7889,0.6987,0.7821,0.7099
1,2024,0.7950,0.6874,0.8018,0.6919


In [23]:

# Cell 21: Create the crop-recommendation classification dataset

# Start with the cleaned historical data and exclude the "Total" season.
df_rec = df_historical_primary_named_seasons.copy()

# Keep only rows with valid crop and context information.
df_rec = df_rec.dropna(
    subset=["State", "District", "Crop", "Season", "start_year", "Yield"]
)

# For each district-season-year, identify the recorded crop with
# the highest observed yield.
group_cols = ["State", "District", "Season", "start_year"]

df_rec = df_rec.sort_values(
    group_cols + ["Yield", "Crop"],
    ascending=[True, True, True, True, False, True]
)

best_crop = (
    df_rec.groupby(group_cols, as_index=False)
    .first()[group_cols + ["Crop", "Yield"]]
    .rename(columns={
        "Crop": "Recommended_Crop",
        "Yield": "Best_Observed_Yield"
    })
)

print("Recommendation dataset shape:", best_crop.shape)
print("Unique target crops:", best_crop["Recommended_Crop"].nunique())
print("\nTarget crop distribution:")
display(best_crop["Recommended_Crop"].value_counts().to_frame("Groups"))

print("\nYear coverage:")
display(best_crop["start_year"].value_counts().sort_index().to_frame("Groups"))

display(best_crop.head())

Recommendation dataset shape: (18440, 6)
Unique target crops: 13

Target crop distribution:


,Groups
Recommended_Crop,
Rice,7024
Maize,5833
Wheat,3670
Bajra,513
Moong,389
Barley,273
Tur,201
Urad,187
Jowar,137



Year coverage:


,Groups
start_year,
2013,1363
2014,1412
2015,1431
2016,1487
2017,1540
2018,1514
2019,1563
2020,1614
2021,1557


,State,District,Season,start_year,Recommended_Crop,Best_Observed_Yield
0,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2013,Rice,2010.0
1,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2014,Rice,2052.0
2,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2015,Rice,2250.0
3,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2016,Rice,2579.0
4,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2017,Rice,2104.0


In [24]:

# Cell 22: Join weather features to the recommendation dataset

weather_cols = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
]

weather_keys = ["State", "District", "Season", "start_year"]

# Use the existing integrated dataset.
df_weather = df.copy() if "df" in globals() else None

if df_weather is None or not set(weather_keys + weather_cols).issubset(df_weather.columns):
    integrated_path = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "weather_regional"
        / "apy_weather_integrated_exploratory.csv"
    )
    df_weather = pd.read_csv(integrated_path, low_memory=False)

# Keep only the required keys and weather columns.
df_weather = df_weather[weather_keys + weather_cols].copy()
df_weather["start_year"] = pd.to_numeric(
    df_weather["start_year"], errors="coerce"
)

# Convert weather values to numeric.
for col in weather_cols:
    df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

# Avoid duplicating recommendation rows during the merge.
df_weather = df_weather.drop_duplicates(subset=weather_keys)

df_rec_model = best_crop.merge(
    df_weather,
    on=weather_keys,
    how="left",
    validate="one_to_one"
)

print("Merged dataset shape:", df_rec_model.shape)
print("\nWeather missing-value percentage:")
display(
    (df_rec_model[weather_cols].isna().mean() * 100)
    .round(2)
    .to_frame("Missing (%)")
)

print("\nSample:")
display(df_rec_model.head())


Merged dataset shape: (18440, 9)

Weather missing-value percentage:


,Missing (%)
tmin_mean_c,14.5
tmax_mean_c,14.5
precipitation_total_mm,14.5



Sample:


,State,District,Season,start_year,Recommended_Crop,Best_Observed_Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm
0,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2013,Rice,2010.0,26.942575,28.726685,3568.43
1,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2014,Rice,2052.0,26.972658,28.891616,2699.38
2,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2015,Rice,2250.0,27.149041,29.036301,2355.34
3,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2016,Rice,2579.0,27.247787,29.249317,2543.72
4,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,2017,Rice,2104.0,27.155014,28.915534,2446.08


In [25]:

# Cell 23: Split data chronologically

TARGET = "Recommended_Crop"

CATEGORICAL_FEATURES = [
    "State",
    "District",
    "Season",
]

WEATHER_FEATURES = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
]

FEATURES = CATEGORICAL_FEATURES + WEATHER_FEATURES

train_df = df_rec_model[df_rec_model["start_year"] <= 2022].copy()
test_df = df_rec_model[df_rec_model["start_year"].isin([2023, 2024])].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training years:", train_df["start_year"].min(), "-", train_df["start_year"].max())
print("Testing years:", sorted(test_df["start_year"].unique()))

print("\nUnique target crops — training:", y_train.nunique())
print("Unique target crops — testing:", y_test.nunique())

unseen_test_crops = sorted(set(y_test) - set(y_train))
print("Target crops absent from training:", unseen_test_crops)


Training rows: 15121
Testing rows: 3319
Training years: 2013 - 2022
Testing years: [np.int64(2023), np.int64(2024)]

Unique target crops — training: 13
Unique target crops — testing: 13
Target crops absent from training: []


In [26]:

# Cell 24: Train and evaluate crop-recommendation classifiers

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            CATEGORICAL_FEATURES,
        ),
        (
            "weather",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            WEATHER_FEATURES,
        ),
    ]
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=15,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Macro F1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Weighted F1": f1_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
).reset_index(drop=True)

display(results_df.round(4))


,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
0,Logistic Regression,0.6189,0.5332,0.3678,0.6555
1,Random Forest,0.5999,0.5027,0.3340,0.6309
2,Decision Tree,0.5658,0.4325,0.3184,0.5963


In [27]:

# Cell 25: Per-crop evaluation of the best model

from sklearn.metrics import classification_report, confusion_matrix

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)

print("Best model:", best_model_name)
print("\nPer-crop classification report:\n")

report = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

crop_report = report_df.drop(
    index=["accuracy", "macro avg", "weighted avg"],
    errors="ignore"
).sort_values("f1-score")

display(crop_report[["precision", "recall", "f1-score", "support"]].round(3))

print("\nMost common actual vs predicted crops:")
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

display(pd.crosstab(
    comparison["Actual"],
    comparison["Predicted"],
    margins=True
))


Best model: Logistic Regression

Per-crop classification report:



,precision,recall,f1-score,support
Lentil,0.000,0.000,0.000,11.0
Small Millets,0.000,0.000,0.000,3.0
Ragi,0.013,0.200,0.024,5.0
Jowar,0.091,0.381,0.147,21.0
Urad,0.124,0.611,0.207,36.0
Tur,0.129,0.864,0.225,22.0
Barley,0.195,0.722,0.308,36.0
Gram,0.325,0.650,0.433,20.0
Bajra,0.506,0.788,0.617,104.0
Maize,0.751,0.534,0.624,1184.0



Most common actual vs predicted crops:


Predicted,Bajra,Barley,Gram,Jowar,Lentil,Maize,Moong,Ragi,Rice,Small Millets,Tur,Urad,Wheat,All
Actual,,,,,,,,,,,,,,
Bajra,82,4,0,2,0,3,6,0,3,2,0,2,0,104
Barley,0,26,0,0,0,0,0,0,0,0,0,0,10,36
Gram,0,0,13,0,0,2,0,0,2,0,0,0,3,20
Jowar,2,1,1,8,0,2,2,0,0,0,1,3,1,21
Lentil,0,0,0,0,0,0,0,0,9,0,0,0,2,11
Maize,22,23,10,43,4,632,29,36,210,9,60,59,47,1184
Moong,3,0,0,3,0,0,109,0,10,0,0,10,0,135
Ragi,0,0,0,0,0,2,0,1,0,2,0,0,0,5
Rice,50,17,0,24,2,189,18,37,636,9,46,81,1,1110


In [28]:

# Cell 26: Compare Logistic Regression with a majority-class baseline

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
logistic_pred = trained_models["Logistic Regression"].predict(X_test)

baseline_results = pd.DataFrame([
    {
        "Model": "Majority-class baseline",
        "Accuracy": accuracy_score(y_test, baseline_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, baseline_pred),
        "Macro F1": f1_score(
            y_test, baseline_pred, average="macro", zero_division=0
        ),
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, logistic_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, logistic_pred),
        "Macro F1": f1_score(
            y_test, logistic_pred, average="macro", zero_division=0
        ),
    },
])

display(baseline_results.round(4))


,Model,Accuracy,Balanced Accuracy,Macro F1
0,Majority-class baseline,0.3344,0.0769,0.0386
1,Logistic Regression,0.6189,0.5332,0.3678


In [29]:

# Cell 27: Inspect available features in the integrated dataset

integrated_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_weather_integrated_exploratory.csv"
)

df_integrated = pd.read_csv(integrated_path, nrows=5)
all_columns = df_integrated.columns.tolist()

keywords = [
    "soil", "ph", "nitrogen", "phosphorus", "potassium",
    "moisture", "irrigation", "water", "humidity", "temperature",
    "precipitation", "rain", "yield", "area", "production",
    "crop", "season", "state", "district", "year"
]

for keyword in keywords:
    matches = [col for col in all_columns if keyword in col.lower()]
    if matches:
        print(f"{keyword}: {matches}")


ph: ['apy_geography_id']
temperature: ['QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature']
precipitation: ['precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'precipitation_valid_days', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation']
yield: ['Yield', 'QC_PositiveYieldWithoutPositiveAreaProduction']
area: ['Area', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_PositiveYieldWithoutPositiveAreaProduction']
production: ['Production', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_PositiveYieldWithoutPositiveAreaProduction']
crop: ['Crop', 'Crop_Category']
season: ['Season', 'weather_full_season_alignment_verified']
state: ['State']
district: ['District', 'QC_Long_District_Grid_Distance']
year: ['Year', 'start_year', 'weather_proxy_calendar_year']


In [30]:

# Cell 28: Inspect existing ranking evaluation results

print("Available ranking-related variables:")

for name in [
    "ranking_metrics",
    "ranking_metrics_by_year",
    "ranking_metrics_by_candidate_count",
    "df_ranked",
    "df_predictions",
    "evaluation_results",
]:
    if name in globals():
        value = globals()[name]
        print(f"\n{name}:")
        if isinstance(value, pd.DataFrame):
            display(value.head())
        elif isinstance(value, dict):
            print(value)
        else:
            print(type(value).__name__, getattr(value, "shape", ""))


Available ranking-related variables:


In [41]:

# Cell 29: Check whether ranking outputs already exist

expected_files = [
    CROP_RECOMMENDATION_DIR / "crop_recommendations_ranked.csv",
    CROP_RECOMMENDATION_DIR / "crop_recommendation_evaluation.json",
    CROP_RECOMMENDATION_DIR / "crop_recommendation_metadata.json",
]

for path in expected_files:
    print(f"{'EXISTS' if path.exists() else 'NOT FOUND'}: {path}")


EXISTS: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\crop_recommendation\crop_recommendations_ranked.csv
EXISTS: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\crop_recommendation\crop_recommendation_evaluation.json
EXISTS: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\crop_recommendation\crop_recommendation_metadata.json


In [32]:

# Cell 30: Inspect Notebook 18 structure

import nbformat

notebook_path = PROJECT_ROOT / "notebooks" / "18_crop_recommendation.ipynb"
nb = nbformat.read(notebook_path, as_version=4)

for i, cell in enumerate(nb.cells):
    if cell.cell_type == "markdown":
        lines = [
            line.strip()
            for line in cell.source.splitlines()
            if line.strip().startswith("#")
        ]
        if lines:
            print(f"Cell {i} [MARKDOWN]: {' | '.join(lines[:3])}")
    elif cell.cell_type == "code":
        first_lines = [
            line for line in cell.source.splitlines()
            if line.strip() and not line.strip().startswith("#")
        ]
        preview = " | ".join(first_lines[:3])
        print(f"Cell {i} [CODE]: {preview[:180]}")


Cell 0 [MARKDOWN]: # Notebook 18: Crop Recommendation Intelligence | ## 1. Objective | ## 2. Scope
Cell 1 [CODE]: _current_path = Path.cwd().resolve() | PROJECT_ROOT = next( |     (
Cell 2 [CODE]: print("Relevant project files:\n") | SEARCH_DIRS = [ |     PROJECT_ROOT / "data",
Cell 3 [CODE]: CANDIDATE_DATASETS = { |     "apy_long": ( |         PROJECT_ROOT
Cell 4 [CODE]: DATA_PATH = ( |     PROJECT_ROOT |     / "data"
Cell 5 [CODE]: year_start_from_label = ( |     df_inspection["Year"] |     .astype(str)
Cell 6 [CODE]: import numpy as np | df_apy = pd.read_csv(DATA_PATH) | required_columns = [
Cell 7 [CODE]: yield_thresholds = df_historical_valid["Yield"].quantile( |     [0.95, 0.99, 0.995, 0.999] | )
Cell 8 [CODE]: quality_flags = [ |     "QC_AreaZero-ProductionPositive", |     "QC_AreaPositive-ProductionZero",
Cell 9 [CODE]: quality_flag_columns = [ |     "QC_AreaZero-ProductionPositive", |     "QC_AreaPositive-ProductionZero",
Cell 10 [CODE]: if not YIELD_MODEL_METADATA_PATH.is_fil

In [40]:

# Cell 31: Rank crop candidates and evaluate ranking quality

from scipy.stats import spearmanr

group_columns = ["State", "District", "Season", "FE_YearStart"]

df_ranked = df_predictions.copy() if "df_predictions" in globals() else None

if df_ranked is None or "Predicted_Yield" not in df_ranked.columns:
    # Reuse the dataframe produced by Cell 19 if it has another name.
    possible_frames = [
        (name, value)
        for name, value in globals().items()
        if isinstance(value, pd.DataFrame)
        and {"State", "District", "Season", "FE_YearStart",
             "Crop", "Yield", "Predicted_Yield"}.issubset(value.columns)
    ]

    if not possible_frames:
        raise NameError(
            "Could not find the prediction dataframe. "
            "Inspect the dataframe variable created in Cell 19."
        )

    df_ranked = possible_frames[0][1].copy()

df_ranked = df_ranked.sort_values(
    group_columns + ["Predicted_Yield", "Crop"],
    ascending=[True, True, True, True, False, True],
).copy()

df_ranked["Predicted_Rank"] = (
    df_ranked.groupby(group_columns)["Predicted_Yield"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df_ranked["Actual_Rank"] = (
    df_ranked.groupby(group_columns)["Yield"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_results = []

for group_key, group in df_ranked.groupby(group_columns):
    if len(group) < 2:
        continue

    actual_best = group["Yield"].max()
    predicted_best = group["Predicted_Yield"].max()

    ranking_results.append({
        "State": group_key[0],
        "District": group_key[1],
        "Season": group_key[2],
        "FE_YearStart": int(group_key[3]),
        "Candidate_Count": len(group),
        "Top1_Hit": int(
            group.loc[group["Predicted_Rank"] == 1, "Yield"].max()
            == actual_best
        ),
        "Top3_Hit": int(
            (group.loc[group["Predicted_Rank"] <= 3, "Yield"] == actual_best).any()
        ),
        "Spearman": spearmanr(
            group["Predicted_Yield"], group["Yield"]
        ).statistic,
    })

ranking_metrics = pd.DataFrame(ranking_results)

print("Groups evaluated:", len(ranking_metrics))
print("Top-1 hit rate:", round(ranking_metrics["Top1_Hit"].mean(), 4))
print("Top-3 hit rate:", round(ranking_metrics["Top3_Hit"].mean(), 4))
print("Mean Spearman correlation:", round(ranking_metrics["Spearman"].mean(), 4))

display(ranking_metrics.head())


C:\Users\gsr58\AppData\Local\Temp\ipykernel_18396\996968061.py:66: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(


Groups evaluated: 2365
Top-1 hit rate: 0.792
Top-3 hit rate: 0.9983
Mean Spearman correlation: 0.792


,State,District,Season,FE_YearStart,Candidate_Count,Top1_Hit,Top3_Hit,Spearman
0,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2023,4,1,1,1.0
1,Andhra Pradesh,Alluri Sitharama Raju,Kharif,2024,4,0,1,0.8
2,Andhra Pradesh,Alluri Sitharama Raju,Rabi,2023,2,1,1,1.0
3,Andhra Pradesh,Alluri Sitharama Raju,Rabi,2024,3,1,1,1.0
4,Andhra Pradesh,Anakapalli,Kharif,2023,2,1,1,1.0


In [35]:

# Cell 32: Ranking evaluation breakdown

print("Performance by evaluation year:")
display(
    ranking_metrics.groupby("FE_YearStart")
    .agg(
        Groups=("Top1_Hit", "size"),
        Mean_Candidates=("Candidate_Count", "mean"),
        Top1_Hit_Rate=("Top1_Hit", "mean"),
        Top3_Hit_Rate=("Top3_Hit", "mean"),
        Mean_Spearman=("Spearman", "mean"),
    )
    .round(4)
)

ranking_metrics["Candidate_Bucket"] = pd.cut(
    ranking_metrics["Candidate_Count"],
    bins=[1, 2, 3, 5, 9, float("inf")],
    labels=["2 crops", "3 crops", "4–5 crops", "6–9 crops", "10+ crops"],
    include_lowest=True,
)

print("\nPerformance by number of candidate crops:")
display(
    ranking_metrics.groupby("Candidate_Bucket", observed=False)
    .agg(
        Groups=("Top1_Hit", "size"),
        Top1_Hit_Rate=("Top1_Hit", "mean"),
        Top3_Hit_Rate=("Top3_Hit", "mean"),
        Mean_Spearman=("Spearman", "mean"),
    )
    .round(4)
)


Performance by evaluation year:


,Groups,Mean_Candidates,Top1_Hit_Rate,Top3_Hit_Rate,Mean_Spearman
FE_YearStart,,,,,
2023,1175,3.1583,0.7889,0.9983,0.7821
2024,1190,3.1664,0.7950,0.9983,0.8018



Performance by number of candidate crops:


,Groups,Top1_Hit_Rate,Top3_Hit_Rate,Mean_Spearman
Candidate_Bucket,,,,
2 crops,887,0.8658,1.0000,0.7314
3 crops,716,0.7849,1.0000,0.8219
4–5 crops,629,0.7266,0.9968,0.8360
6–9 crops,133,0.6466,0.9850,0.8266
10+ crops,0,NaN,NaN,NaN


In [37]:

# Replace the evaluation_summary creation and JSON-writing section

import json
import numpy as np

def make_json_safe(value):
    """Convert pandas/NumPy values and NaN/Infinity to JSON-safe values."""
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.bool_):
        return bool(value)
    if pd.isna(value):
        return None
    return value

evaluation_summary = make_json_safe(evaluation_summary)

with open(
    CROP_RECOMMENDATION_DIR / "crop_recommendation_evaluation.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(evaluation_summary, f, indent=2, allow_nan=False)

print("Evaluation JSON saved successfully.")


Evaluation JSON saved successfully.


In [38]:

# Cell 34: Verify all saved output files

expected_files = [
    CROP_RECOMMENDATION_DIR / "crop_recommendations_ranked.csv",
    CROP_RECOMMENDATION_DIR / "crop_recommendation_evaluation.json",
    CROP_RECOMMENDATION_DIR / "crop_recommendation_metadata.json",
]

for path in expected_files:
    print(f"{path.name}: {'EXISTS' if path.is_file() else 'MISSING'}")

if all(path.is_file() for path in expected_files):
    ranked_check = pd.read_csv(expected_files[0])

    with open(expected_files[1], encoding="utf-8") as f:
        evaluation_check = json.load(f)

    with open(expected_files[2], encoding="utf-8") as f:
        metadata_check = json.load(f)

    print("\nRanked CSV shape:", ranked_check.shape)
    print("Evaluation groups:", evaluation_check["evaluated_groups"])
    print("Top-1 hit rate:", evaluation_check["top1_hit_rate"])
    print("Model:", metadata_check["model_name"])
    print("Prediction year configured:", metadata_check["prediction_year_configured"])
    print("\nAll three files exist and both JSON files are valid.")


crop_recommendations_ranked.csv: EXISTS
crop_recommendation_evaluation.json: EXISTS
crop_recommendation_metadata.json: MISSING


In [39]:

# Repair: create the missing metadata JSON

import json
from datetime import datetime, timezone

metadata = {
    "project": "BharatCrop Intelligence",
    "model_name": yield_model_metadata.get(
        "model_name", "HistGradientBoostingRegressor"
    ),
    "model_artifact": str(YIELD_MODEL_PATH.relative_to(PROJECT_ROOT)),
    "target": "Yield",
    "recommendation_method": (
        "Rank eligible historical crop candidates by predicted yield "
        "within State-District-Season-Year groups."
    ),
    "prediction_year_configured": PREDICTION_YEAR,
    "evaluation_years": [2023, 2024],
    "features": yield_model_metadata.get("features", []),
    "limitations": [
        "The saved ranking CSV contains retrospective rankings for 2023–2024.",
        "Only historically observed and eligible crop candidates are included.",
        "Predicted yield does not establish profitability or agronomic suitability.",
        "Soil pH, NPK, soil moisture, and irrigation features are unavailable.",
        "Weather mapping and full-season alignment are not verified.",
        "The selected model excludes weather features.",
        "2025 predictions are extrapolations beyond the 2013–2022 training period.",
        "Historical ranking performance does not establish causal effects or production readiness."
    ],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}

metadata_path = (
    CROP_RECOMMENDATION_DIR / "crop_recommendation_metadata.json"
)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, allow_nan=False)

print("Metadata saved:", metadata_path.is_file())


Metadata saved: True


In [42]:

# Cell 30: Inspect the saved historical rankings

ranked_path = (
    CROP_RECOMMENDATION_DIR / "crop_recommendations_ranked.csv"
)

df_saved_rankings = pd.read_csv(ranked_path)

print("Saved ranking shape:", df_saved_rankings.shape)
print("\nColumns:")
print(df_saved_rankings.columns.tolist())

print("\nYear values:")
year_col = next(
    (c for c in ["FE_YearStart", "Year", "start_year"]
     if c in df_saved_rankings.columns),
    None,
)
if year_col:
    print(sorted(df_saved_rankings[year_col].dropna().unique().tolist()))
else:
    print("No recognizable year column found.")

print("\nSample rows:")
display(df_saved_rankings.head(5))


Saved ranking shape: (8433, 43)

Columns:
['State', 'District', 'Crop', 'Season', 'Crop_Category', 'Year', 'Area', 'Production', 'Yield', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_NegativeMeasurements', 'QC_PositiveYieldWithoutPositiveAreaProduction', 'apy_geography_id', 'start_year', 'weather_proxy_calendar_year', 'tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days', 'QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation', 'QC_Long_District_Grid_Distance', 'weather_proxy_available', '_merge', 'apy_measurements_complete', 'weather_and_apy_complete', 'weather_alignment_method', 'weather_full_season_alignment_verified', 'weather_available', 'weather_match_status', 'FE_YearStart', 'Predicted_Yield', 'Predicted_Rank', 'Actual_Ra

,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction,apy_geography_id,start_year,weather_proxy_calendar_year,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance,weather_proxy_available,_merge,apy_measurements_complete,weather_and_apy_complete,weather_alignment_method,weather_full_season_alignment_verified,weather_available,weather_match_status,FE_YearStart,Predicted_Yield,Predicted_Rank,Actual_Rank
0,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Kharif,Individual Crop Candidate,2023-24,0.05,0.10,2100.0,False,False,False,False,APY-GEO-0002,2023,2023.0,27.310932,23.64,29.326932,32.56,2274.02,6.230192,83.96,365.0,365.0,365.0,False,False,False,False,True,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED,2023,2435.288112,1,1
1,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Kharif,Individual Crop Candidate,2024-25,0.04,0.09,2062.0,False,False,False,False,APY-GEO-0002,2024,2024.0,27.557404,24.11,29.604208,34.11,2214.15,6.049590,81.64,366.0,366.0,366.0,False,False,False,False,True,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED,2024,2435.288112,1,1
2,Andhra Pradesh,Alluri Sitharama Raju,Maize,Kharif,Individual Crop Candidate,2023-24,0.02,0.06,3438.0,False,False,False,False,APY-GEO-0004,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,True,False,STARTING_CALENDAR_YEAR_PROXY,False,False,NO_WEATHER_COVERAGE,2023,4294.018903,1,1
3,Andhra Pradesh,Alluri Sitharama Raju,Rice,Kharif,Individual Crop Candidate,2023-24,0.52,0.92,1762.0,False,False,False,False,APY-GEO-0004,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,True,False,STARTING_CALENDAR_YEAR_PROXY,False,False,NO_WEATHER_COVERAGE,2023,3449.962015,2,2
4,Andhra Pradesh,Alluri Sitharama Raju,Ragi,Kharif,Individual Crop Candidate,2023-24,0.20,0.25,1269.0,False,False,False,False,APY-GEO-0004,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,True,False,STARTING_CALENDAR_YEAR_PROXY,False,False,NO_WEATHER_COVERAGE,2023,1552.285840,3,3


In [43]:

# Cell 31: Generate separate 2025 projections from historically observed candidates

import joblib
import numpy as np
import pandas as pd

PREDICTION_YEAR = 2025

# Use eligible historical records to define which crop candidates have evidence.
df_candidates = df_historical_primary_named_seasons.copy()

df_candidates["FE_YearStart"] = pd.to_numeric(
    df_candidates["start_year"], errors="coerce"
)
df_candidates = df_candidates.dropna(
    subset=["State", "District", "Crop", "Season", "FE_YearStart"]
)

# Keep only combinations observed in the historical data.
candidate_columns = ["State", "District", "Season", "Crop", "Crop_Category"]
df_2025 = (
    df_candidates[candidate_columns]
    .drop_duplicates()
    .copy()
)

df_2025["FE_YearStart"] = PREDICTION_YEAR

# Load the existing selected model; do not retrain.
yield_model = joblib.load(YIELD_MODEL_PATH)

model_features = yield_model_metadata["features"]
df_2025["Predicted_Yield"] = yield_model.predict(
    df_2025[model_features]
)

if not np.isfinite(df_2025["Predicted_Yield"]).all():
    raise ValueError("Non-finite predictions found; refusing to save projections.")

group_columns = ["State", "District", "Season", "FE_YearStart"]

df_2025["Predicted_Rank"] = (
    df_2025.groupby(group_columns)["Predicted_Yield"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df_2025 = df_2025.sort_values(
    group_columns + ["Predicted_Rank", "Crop"]
).reset_index(drop=True)

projection_path = (
    CROP_RECOMMENDATION_DIR / "crop_recommendations_projection_2025.csv"
)
df_2025.to_csv(projection_path, index=False)

print("Projection saved:", projection_path.is_file())
print("Prediction year:", PREDICTION_YEAR)
print("Candidate rows:", len(df_2025))
print(
    "District-season groups:",
    df_2025[group_columns].drop_duplicates().shape[0]
)
print("\nTop-ranked projections:")
display(
    df_2025[df_2025["Predicted_Rank"] == 1].head(10)
)


Projection saved: True
Prediction year: 2025
Candidate rows: 6220
District-season groups: 1850

Top-ranked projections:


,State,District,Season,Crop,Crop_Category,FE_YearStart,Predicted_Yield,Predicted_Rank
0,Andaman And Nicobar Islands,North And Middle Andaman,Kharif,Rice,Individual Crop Candidate,2025,2435.288112,1
1,Andaman And Nicobar Islands,North And Middle Andaman,Rabi,Moong,Individual Crop Candidate,2025,803.777318,1
2,Andhra Pradesh,Alluri Sitharama Raju,Kharif,Maize,Individual Crop Candidate,2025,4294.018903,1
7,Andhra Pradesh,Alluri Sitharama Raju,Rabi,Maize,Individual Crop Candidate,2025,7199.436163,1
10,Andhra Pradesh,Anakapalli,Kharif,Rice,Individual Crop Candidate,2025,3449.962015,1
12,Andhra Pradesh,Anakapalli,Rabi,Rice,Individual Crop Candidate,2025,3790.162458,1
15,Andhra Pradesh,Ananthapuramu,Kharif,Maize,Individual Crop Candidate,2025,3856.279936,1
24,Andhra Pradesh,Ananthapuramu,Rabi,Maize,Individual Crop Candidate,2025,6761.697196,1
32,Andhra Pradesh,Annamayya,Kharif,Maize,Individual Crop Candidate,2025,4294.018903,1
35,Andhra Pradesh,Annamayya,Rabi,Maize,Individual Crop Candidate,2025,7199.436163,1


In [44]:

# Cell 32: Projection sanity checks

print("Projection rows:", len(df_2025))
print("Unique crops:", df_2025["Crop"].nunique())
print("Unique districts:", df_2025["District"].nunique())
print("Unique district-season groups:",
      df_2025[group_columns].drop_duplicates().shape[0])

print("\nPredicted yield distribution:")
display(df_2025["Predicted_Yield"].describe().to_frame())

print("\nMost frequent predicted yields:")
display(
    df_2025["Predicted_Yield"]
    .round(3)
    .value_counts()
    .head(10)
    .rename_axis("Predicted_Yield")
    .to_frame("Count")
)

print("\nTop-ranked crop counts:")
display(
    df_2025[df_2025["Predicted_Rank"] == 1]["Crop"]
    .value_counts()
    .rename_axis("Crop")
    .to_frame("Top_Rank_Groups")
)


Projection rows: 6220
Unique crops: 13
Unique districts: 741
Unique district-season groups: 1850

Predicted yield distribution:


,Predicted_Yield
count,6220.000000
mean,2022.876388
std,1265.224450
min,136.011866
25%,1040.752311
50%,1716.737770
75%,2716.682196
max,8557.188649



Most frequent predicted yields:


,Count
Predicted_Yield,
2702.200,27
3633.397,27
2209.324,26
2836.185,24
1082.898,23
695.519,23
1895.907,23
2844.374,23
3357.297,22



Top-ranked crop counts:


,Top_Rank_Groups
Crop,
Rice,735
Maize,651
Wheat,323
Moong,77
Bajra,42
Barley,7
Gram,5
Jowar,3
Small Millets,3


In [45]:

# Cell 33: Inspect district-level recommendation diversity

top_recommendations = df_2025[
    df_2025["Predicted_Rank"] == 1
].copy()

print("Top-ranked recommendation rows:", len(top_recommendations))
print("Unique top-ranked crops:", top_recommendations["Crop"].nunique())

print("\nTop crop share:")
crop_share = (
    top_recommendations["Crop"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Share_Percent")
    .to_frame()
)
display(crop_share.head(10).round(2))

print("\nNumber of distinct top crops by season:")
display(
    top_recommendations.groupby("Season")["Crop"]
    .nunique()
    .rename("Distinct_Top_Crops")
    .to_frame()
)

print("\nTop crop counts by season:")
display(
    pd.crosstab(
        top_recommendations["Season"],
        top_recommendations["Crop"]
    )
)

Top-ranked recommendation rows: 1850
Unique top-ranked crops: 12

Top crop share:


,Share_Percent
Crop,
Rice,39.73
Maize,35.19
Wheat,17.46
Moong,4.16
Bajra,2.27
Barley,0.38
Gram,0.27
Jowar,0.16
Small Millets,0.16



Number of distinct top crops by season:


,Distinct_Top_Crops
Season,
Kharif,6
Rabi,9
Summer,5



Top crop counts by season:


Crop,Bajra,Barley,Gram,Jowar,Maize,Moong,Ragi,Rice,Small Millets,Tur,Urad,Wheat
Season,,,,,,,,,,,,
Kharif,25,0,0,2,271,0,0,436,2,1,0,0
Rabi,0,7,5,1,236,1,1,111,1,0,0,323
Summer,17,0,0,0,144,76,0,188,0,0,2,0



## Final Results and Limitations

### Outputs
- `models/crop_recommendation/crop_recommendations_ranked.csv` — retrospective rankings evaluated on 2023–2024 observations.
- `models/crop_recommendation/crop_recommendation_evaluation.json` — historical ranking metrics.
- `models/crop_recommendation/crop_recommendation_metadata.json` — model configuration and limitations.
- `models/crop_recommendation/crop_recommendations_projection_2025.csv` — yield-based projections for historically observed crop candidates.

### Historical Evaluation
- Evaluated district–season–year groups: 2,365.
- Top-1 hit rate: 79.20%.
- Top-3 hit rate: 99.83%; interpret cautiously because most groups contain few candidates.
- Mean Spearman correlation: 0.792.

### Projection Diagnostics
- Projection candidates: 6,220.
- District–season groups: 1,850.
- Historically observed crops represented: 13.
- Rice, Maize, and Wheat account for approximately 92.38% of top-ranked projections.

### Limitations
1. The model predicts yield, not profitability or complete agronomic suitability.
2. Projections use a 2025 year value, beyond the 2013–2022 training period; they are extrapolations, not validated 2025 outcomes.
3. Candidate crops are restricted to historically observed eligible combinations.
4. Weather is not included in the selected model.
5. Weather mapping and full-season alignment remain unverified.
6. Recommendations do not incorporate validated soil, irrigation, crop-duration, market-price, or input-cost constraints.
7. Historical ranking metrics do not establish causal effects or guarantee future performance.

### Conclusion
Notebook 18 produces historical ranking evaluations and a separate 2025 yield-based projection artifact. These outputs are suitable for exploratory analysis, not for unqualified farm-level planting advice.
